# TODO:

Politics / Mentions: two promising wallets
0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8	
0x0cb10c40b0776e9ee8cef970af85724654dda76c


# Wallet Strategy Selection

This stage tracks two distinct wallet groups on the full trade stream (BUY + SELL):

- `copyable_group`: wallets that pass base eligibility and strong copyability thresholds.
- `predicting_group`: the remaining eligible wallets (not in copyable) that pass a predictability score threshold.

Notes:
- Group membership is threshold-based (absolute scores), not target-count based.
- Open-buy skill metrics are kept for diagnostics/legacy comparison.
- `selected_wallets_v2.parquet` stores both groups with a `wallet_group` label.
- Stage2 backtest uses only `predicting_group`.


In [52]:
%load_ext autoreload
%autoreload 2

import datetime
from pathlib import Path
import json

import numpy as np
import pandas as pd
import pyarrow.dataset as ds

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.2f}'.format

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Configuration

In [53]:
from enum import Enum

class MarketFilterMode(Enum):
    DEFAULT = "default"
    SPORTS = "sports"
    CRYPTO = "crypto"

# ── MARKET FILTER MODE SWITCH ────────────────────────────────────────────────
MARKET_FILTER_MODE = MarketFilterMode.SPORTS

# ── configuration ─────────────────────────────────────────────────────────────
# TAGS = set(['Weather'])
# TAGS = set(['Esports'])
TAGS = set(['Sports'])
# TAGS = set(['Awards'])
# TAGS = set(['MrBeast', 'Trump', 'Elections', 'Celebrities', 'Elon Musk', 'YouTube'])

RECENCY_DAYS = 90

MAX_LEAD_DAYS = 2
if 'Politics' in TAGS:
    MAX_LEAD_DAYS = 14

PRICE_BUCKET_BINS   = [0.0, 0.1, 0.25, 0.4, 0.6, 0.75, 0.9, 1.0]
PRICE_BUCKET_LABELS = [
    "0.00-0.10", "0.10-0.25", "0.25-0.40", "0.40-0.60",
    "0.60-0.75", "0.75-0.90", "0.90-1.00",
]

if MARKET_FILTER_MODE == MarketFilterMode.SPORTS:
    TRADES_DIR    = Path("../../data/polygon_trades_processed_sports")
    WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2_sports")
elif MARKET_FILTER_MODE == MarketFilterMode.CRYPTO:
    TRADES_DIR    = Path("../../data/polygon_trades_processed_crypto")
    WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2_crypto")
else:
    TRADES_DIR    = Path("../../data/polygon_trades_processed")
    WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2")
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

dataset = ds.dataset(TRADES_DIR, format="parquet")


WALLET_FILTER = None
# WALLET_FILTER = [
#     '0x919698b19427cbe6945b0dc823f2d9e126a4d934',
#     '0x85d22a973e817f87afe0a7fd346a39823cce7479',
#     '0x6011655c4afb76f36dd1b08a137a1ba73466b31e',
#     '0x5f612351a9e46afbe3f164473a7d77a69ec9840a',
# ]
# ── exposure thresholds (bankroll-discipline carve, signal_lab §14-18) ─────
# Copy only trades whose per-wallet exposure (position * price) is >= T_{w,f},
# the largest exposure capturing fraction `f` of the wallet's train copyable PnL
# (fit via wallet_capture_thresholds on train; f == 1.0 copies everything).
EXPOSURE_FRACS   = (1, 0.5, 0.2) #, 0.95, 0.9, 0.8, 0.5, 0.2, 0.1, 0.05)
EXPOSURE_PNL_COL = "copyable_pnl"
EXPOSURE_EXP_COL = "exposure"   # computed as position * price

# ── early exposure carve (apply filter right after loading) ────────────────
# When set (typically one of EXPOSURE_FRACS, e.g. 0.5), per-wallet capture
# thresholds T_{w,f} are fit on the wallet's train BUY fills via
# wallet_capture_thresholds, and ALL fills (BUY + SELL, train + test) with
# deployed_capital (position * price) < T are dropped from df_full before
# the carved metrics/selection below are computed. Full-stream (uncarved)
# wallet stats are also computed BEFORE the carve, so both stat sets are
# available. Wallets with no positive train copyable_pnl get no threshold
# and are kept whole. None disables the filter (carved == full stats).
EXPOSURE_FILTER_FRAC = 0.2

# ── per-trade exposure cap ───────────────────────────────────────────────────
# Cap on capital deployed per copied trade: EXPOSURE_CAP_FRAC × the wallet's
# BUY-only train PnL (full-stream stats). Trades with notional above the cap
# contribute only the capped fraction of their PnL (pnl_limited /
# copyable_pnl_limited, computed in the load cell).
EXPOSURE_CAP_FRAC = 0.10


DATASET_COLUMNS        = set(dataset.schema.names)
TRIGGER_TX_HASH_COLUMN = (
    "transaction_hash" if "transaction_hash" in DATASET_COLUMNS
    else ("tx_hash" if "tx_hash" in DATASET_COLUMNS else None)
)


## Derive train/test dates from data

## Full-stream wallet metrics for copy-trigger selection

Build per-wallet metrics on the full training trade stream (BUY + SELL).
Copyability is considered here (stage1), not in stage0 filtering.


In [54]:
from polymarket_analysis.wallet_selection.volatility import (
    compute_wallet_metrics,
    filter_wallets_by_volatility,
)

from polymarket_analysis.data.data_catalogue import load_markets_processed
mdf = load_markets_processed()

print(mdf.groupby('primary_tag').size().sort_values(ascending=False).head(20))

_tagset = set(TAGS)
mdf = mdf[
    (True if TAGS is None else mdf['tags'].apply(lambda tags: bool(_tagset & set(tags))))
    & ~(mdf['primary_tag'].isin(['Crypto']))
    & ((TAGS is None) or mdf["tags"].apply(lambda tags: any(tag in TAGS for tag in tags)))
    & (mdf['winner_token_id'].notna())
].copy().reset_index(drop=True)

print(f'selected markets for tags {TAGS}: {len(mdf)}')

primary_tag
Sports         3678528
Crypto         1082612
Weather         191014
Politics        103129
Finance          58994
Awards           12364
Movies           11521
AI                8498
Music             7778
Culture           6947
All               3808
Esports           3699
Business          3452
MrBeast           2302
Science           1868
Iran              1788
Celebrities       1447
Trump             1259
Tech              1018
Elections          928
dtype: int64
selected markets for tags {'Sports'}: 3265838


In [55]:
# Load full processed trades for the full-stream path
trade_files = sorted(TRADES_DIR.glob("*.parquet"))
df_full = pd.concat(
    [pd.read_parquet(f).merge(mdf, on="condition_id", how="inner") for f in trade_files], ignore_index=True
    )

print(f"Full loaded df_full size: {len(df_full)}")

# df_full = df_full.merge(mdf, on="condition_id", how="inner")

df_full['outcome'] = df_full['outcome_x']
del df_full['outcome_x'], df_full['outcome_y']

df_full["dt"] = pd.to_datetime(df_full["dt"], utc=True)

print(f"rows after mdf merge (BUY + SELL): {len(df_full):,}")

lead = (
    pd.to_datetime(df_full["last_condition_trade_ts"], utc=True, errors="coerce")
    - df_full["dt"]
)
df_full = df_full[lead <= pd.Timedelta(days=MAX_LEAD_DAYS)].copy()
print(
    f"Lead filter (<= {MAX_LEAD_DAYS}d before resolution): "
    f"{len(df_full):,} trades"
)

# Normalise grouped schema → canonical fill-level names
if "total_quantity" in df_full.columns and "quantity" not in df_full.columns:
    df_full = df_full.rename(columns={
        "total_quantity": "quantity",
        "avg_price": "price",
        "trade_value_usdc": "usdc_amount",
    })

df_full["usdc_amount"] = df_full["usdc_amount"].astype(float)
df_full["final_value_usdc"] = df_full["final_value_usdc"].astype(float)
df_full["quantity"] = df_full["quantity"].astype(float)

# Full-stream PnL / notional across BUY + SELL
df_full["pnl"] = np.where(
    df_full["side"] == "BUY",
    df_full["final_value_usdc"] - df_full["usdc_amount"],
    df_full["usdc_amount"] - df_full["final_value_usdc"],
)
df_full["notional"] = np.where(
    df_full["side"] == "BUY",
    df_full["usdc_amount"],
    df_full["quantity"] * (1 - df_full["price"].astype(float)),
)
TRAIN_END = pd.Timestamp("2026-08-01", tz="UTC")
df_full['is_train'] = df_full['last_condition_trade_ts'] <= TRAIN_END

_n_wallets = df_full['wallet'].nunique()
_train = df_full[df_full['is_train']]
_test_wallets = set(df_full.loc[~df_full['is_train'], 'wallet'].unique())
_train_stats = _train.groupby('wallet').agg(
    train_pnl=('pnl', 'sum'),
    train_markets=('condition_id', 'nunique'),
)
del _train
_eligible = _train_stats.index[
    _train_stats.index.isin(_test_wallets)
    & (_train_stats['train_pnl'] >= 1000)
    & (_train_stats['train_markets'] >= 20)
]
df_full = df_full[df_full['wallet'].isin(_eligible)].copy()
print(f"Basic wallet filter: {df_full['wallet'].nunique():,} of {_n_wallets:,} wallets kept")

# FULL (uncarved) wallet metrics — computed before the exposure carve so
# both stat sets exist: full stats are passed to the selection filters,
# carved stats keep driving the execution/diagnostics stream below.
df_all_train_full = df_full[df_full["is_train"]].copy()
wallet_vol_all_full, _ = compute_wallet_metrics(df_all_train_full)
del df_all_train_full

df_buy_train_full = df_full[df_full["is_train"] & (df_full["side"] == "BUY")].copy()
wallet_vol_train_full, _ = compute_wallet_metrics(df_buy_train_full)
del df_buy_train_full

wallet_vol_train_full = wallet_vol_train_full.merge(
    wallet_vol_all_full[["wallet", "max_drawdown", "total_pnl"]].rename(
        columns={"max_drawdown": "max_drawdown_all", "total_pnl": "total_pnl_all"}
    ),
    on="wallet",
    how="left",
)
wallet_vol_train_full["max_drawdown_all_to_pnl_all"] = np.where(
    wallet_vol_train_full["total_pnl_all"] > 0,
    wallet_vol_train_full["max_drawdown_all"] / wallet_vol_train_full["total_pnl_all"],
    float("nan"),
)

# Per-wallet exposure cap: EXPOSURE_CAP_FRAC × BUY-only train PnL from the
# full (uncarved) stats — max capital deployed on any single copied trade.
# Non-positive BUY train PnL (or no BUY train fills) → cap 0: nothing deployable.
_cap = EXPOSURE_CAP_FRAC * wallet_vol_train_full.set_index("wallet")["total_pnl"]
df_full["deployed_capital_limit"] = df_full["wallet"].map(_cap).fillna(0.0).clip(lower=0.0)

# PnL limited by the cap: take only the fraction of each trade whose notional
# fits below deployed_capital_limit (notional = capital size of the trade).
_cap_frac = np.where(
    df_full["notional"] > 0,
    np.minimum(1.0, df_full["deployed_capital_limit"] / df_full["notional"]),
    1.0,  # zero-notional trade lies entirely below the cap
)
df_full["deployed_capital_frac"] = _cap_frac
df_full["pnl_limited"] = df_full["pnl"] * _cap_frac
df_full["copyable_pnl_limited"] = df_full["copyable_pnl"] * _cap_frac

# Lean pre-carve BUY stream handed to the explorer: same rows as `df_full`
# before the exposure carve, flagged by `is_carved`, so the PnL panels can
# overlay full and carved variants.
_full_stream_cols = [
    "wallet", "dt", "side", "last_condition_trade_ts", "position", "price",
    "notional", "pnl", "pnl_limited", "copyable_pnl", "copyable_pnl_limited",
    "deployed_capital_frac", "is_train", "copyable_qty_5m_100", "is_carved",
]

if EXPOSURE_FILTER_FRAC is not None:
    from polymarket_analysis.wallet_selection.exposure import wallet_capture_thresholds

    df_full["deployed_capital"] = df_full["position"] * df_full["price"]

    _fit = df_full[df_full["is_train"] & (df_full["side"] == "BUY")]
    _capture = wallet_capture_thresholds(
        _fit, (EXPOSURE_FILTER_FRAC,), EXPOSURE_PNL_COL, "deployed_capital"
    )
    del _fit
    _T = df_full["wallet"].map(_capture.set_index("wallet")["threshold"]).fillna(0.0)

    _keep = df_full["deployed_capital"].notna() & (df_full["deployed_capital"] >= _T)
    df_full["is_carved"] = _keep

    df_full_buy_full = df_full.loc[
        df_full["side"] == "BUY", _full_stream_cols
    ].copy()

    _n = len(df_full)
    df_full = df_full[_keep].copy()
    print(
        f"Exposure carve (frac {EXPOSURE_FILTER_FRAC:g}): "
        f"{len(df_full):,} of {_n:,} trades kept"
    )
else:
    df_full["is_carved"] = True
    df_full_buy_full = df_full.loc[
        df_full["side"] == "BUY", _full_stream_cols
    ].copy()

# TRAIN_START = TRAIN_END - pd.Timedelta(days=30)
# df_full = df_full[df_full['dt'] >= TRAIN_START]

# Carved all-trades (BUY + SELL) wallet metrics: drawdown incl. SELL legs
df_all_train = df_full[df_full["is_train"]].copy()
wallet_vol_all, _ = compute_wallet_metrics(df_all_train)
del df_all_train

# Primary stream stays BUY-only (carved execution/diagnostics stream)
df_full = df_full[df_full["side"] == "BUY"].copy().reset_index(drop=True)

df_slice = df_full[df_full["is_train"]].copy()

wallet_vol_train, _ = compute_wallet_metrics(df_slice)
del df_slice

wallet_vol_train = wallet_vol_train.merge(
    wallet_vol_all[["wallet", "max_drawdown", "total_pnl"]].rename(
        columns={"max_drawdown": "max_drawdown_all", "total_pnl": "total_pnl_all"}
    ),
    on="wallet",
    how="left",
)
wallet_vol_train["max_drawdown_all_to_pnl_all"] = np.where(
    wallet_vol_train["total_pnl_all"] > 0,
    wallet_vol_train["max_drawdown_all"] / wallet_vol_train["total_pnl_all"],
    float("nan"),
)
print(len(wallet_vol_train))
print(len(wallet_vol_train_full))


Full loaded df_full size: 25212960
rows after mdf merge (BUY + SELL): 25,212,960
Lead filter (<= 2d before resolution): 24,384,903 trades
Basic wallet filter: 2,495 of 15,495 wallets kept


/Users/vobornij/projects/polymarket/src/polymarket_analysis/wallet_selection/volatility.py:190: FutureWarning:

The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior

/Users/vobornij/projects/polymarket/src/polymarket_analysis/wallet_selection/volatility.py:190: FutureWarning:

The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior



Exposure carve (frac 0.2): 1,999,197 of 14,905,625 trades kept


/Users/vobornij/projects/polymarket/src/polymarket_analysis/wallet_selection/volatility.py:190: FutureWarning:

The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior

/Users/vobornij/projects/polymarket/src/polymarket_analysis/wallet_selection/volatility.py:190: FutureWarning:

The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior



2495
2495


In [56]:
daily_stats = (
    df_full.groupby(pd.Grouper(key="dt", freq="D"))
    .agg(
        trade_count=("price", "size"),
        total_volume=("price", lambda s: (s * df_full.loc[s.index, "quantity"]).sum()),
    )
    .reset_index()
)

# import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.graph_objects as go


# Create plot with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=daily_stats["dt"],
        y=daily_stats["trade_count"],
        name="Number of trades",
        mode="lines+markers",
    ),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=daily_stats["dt"],
        y=daily_stats["total_volume"],
        name="Total volume (price × qty)",
        mode="lines+markers",
    ),
    secondary_y=True,
)

fig.update_layout(
    title="Daily Trade Count and Total Volume",
    xaxis_title="Date",
    template="plotly_white",
    hovermode="x unified",
)

fig.update_yaxes(title_text="Number of trades", secondary_y=False)
fig.update_yaxes(title_text="Total volume", secondary_y=True)

fig.show()


In [57]:
len(df_full[df_full["is_train"]])

1125137

In [58]:
# mdf = load_markets_processed()
# df_full = pd.concat(
#     [
#         (
#             pd.read_parquet(f)
#             .merge(mdf, on="condition_id", how="inner")
#             .assign(dt=lambda df: pd.to_datetime(df["dt"], utc=True))
#             .loc[lambda df: df["dt"] >=  datetime.datetime(2026, 7, 1, tzinfo=datetime.timezone.utc)]
#         )
#         for f in trade_files
#     ],
#     ignore_index=True,
# )


In [59]:
print (df_full.columns)
print(df_full.index.names)

Index(['wallet', 'condition_id', 'token_id', 'dt', 'side', 'position',
       'quantity', 'price', 'usdc_amount', 'final_value_usdc', 'trade_pnl',
       'copyable_pnl', 'token_winner', 'final_price',
       'last_condition_trade_ts', 'tx_hash', 'num_fills', 'is_train',
       'copyable_qty_5m_100', 'avail_copy_qty_5m_100',
       'avail_copy_total_vol_5m_100', 'avail_copy_count_5m_100',
       'end_date_iso', 'question', 'tags', 'primary_tag', 'winner_token_id',
       'outcome', 'pnl', 'notional', 'deployed_capital_limit',
       'deployed_capital_frac', 'pnl_limited', 'copyable_pnl_limited',
       'deployed_capital', 'is_carved'],
      dtype='object')
[None]


In [60]:
_copy_scale = np.minimum(2, df_full['avail_copy_qty_5m_100'] / df_full['quantity'])
df_full['copyable_qty_2x'] = df_full['copyable_qty_5m_100'] * _copy_scale
df_full['copyable_pnl_2x'] = df_full['copyable_pnl'] * _copy_scale

_copy_scale = np.minimum(0.5, df_full['avail_copy_qty_5m_100'] / df_full['quantity'])

df_full['copyable_qty_0.5x'] = df_full['copyable_qty_5m_100'] * _copy_scale
df_full['copyable_pnl_0.5x'] = df_full['copyable_pnl'] * _copy_scale

In [61]:
# ── Copyable PnL variants ─────────────────────────────────────────────────────
# Per-fill copyable PnL under different lookback-window / fill-price assumptions.
# stage0 exports per-fill qty columns only for the variants enabled in its
# VARIANTS config (currently just the 5m primary); keep only the variants
# whose qty column is present in the current dataset.
COPYABLE_PNL_BASE = "copyable_pnl"
COPYABLE_VARIANTS = {
    # "5m_095": dict(qty_col="copyable_qty_5m_095", fill_factor=0.95),
    # "20m_100": dict(qty_col="copyable_qty_20m_100", fill_factor=1.0),
    # "2x": dict(qty_col="copyable_qty_2x", fill_factor=1.0),
    # "0.5x": dict(qty_col="copyable_qty_0.5x", fill_factor=1.0),
}
COPYABLE_VARIANTS = {
    k: cfg for k, cfg in COPYABLE_VARIANTS.items() if cfg["qty_col"] in df_full.columns
}
print(f"copyable PnL variants in use: {list(COPYABLE_VARIANTS)}")

for key, cfg in COPYABLE_VARIANTS.items():
    qty = df_full[cfg["qty_col"]]
    df_full[f"copyable_pnl_{key}"] = np.where(
        df_full["side"] == "BUY",
        (df_full["final_price"] - df_full["price"] * cfg["fill_factor"]) * qty,
        (df_full["price"] / cfg["fill_factor"] - df_full["final_price"]) * qty,
    )

COPYABLE_PNL_COLS = [COPYABLE_PNL_BASE] + [f"copyable_pnl_{k}" for k in COPYABLE_VARIANTS]
COPYABLE_PNL_VARIANTS = COPYABLE_PNL_COLS[1:]

# Exposure-capped PnL columns (per-wallet deployed_capital_limit, load cell):
# wallet PnL / copyable PnL limited to the fraction of each trade whose
# notional fits below the cap.
CAPPED_PNL_COLS = ["pnl_limited", "copyable_pnl_limited"]


copyable PnL variants in use: []


In [62]:
df_full['copyable_notional'] = df_full['notional'] * (df_full['copyable_pnl'] / df_full['pnl'].replace(0, np.nan))
df_full['roi'] = df_full['pnl'] / df_full['notional'].replace(0, np.nan)
df_full['copyable_roi'] = df_full['copyable_pnl'] / df_full['copyable_notional'].replace(0, np.nan)

df_full.head()

,wallet,condition_id,token_id,dt,side,position,quantity,price,usdc_amount,final_value_usdc,trade_pnl,copyable_pnl,token_winner,final_price,last_condition_trade_ts,tx_hash,num_fills,is_train,copyable_qty_5m_100,avail_copy_qty_5m_100,avail_copy_total_vol_5m_100,avail_copy_count_5m_100,end_date_iso,question,tags,primary_tag,winner_token_id,outcome,pnl,notional,deployed_capital_limit,deployed_capital_frac,pnl_limited,copyable_pnl_limited,deployed_capital,is_carved,copyable_qty_2x,copyable_pnl_2x,copyable_qty_0.5x,copyable_pnl_0.5x,copyable_notional,roi,copyable_roi
0,0x4fc19a907012ef58717ee7ceb1f89057f71aec92,0x0f11f83fb8e412686efcb9430fcbeb326e91f24f4f90539070a3912cf6c31724,40959390651531425935743766952118482746316220645668151461809801143476988026290,2025-01-01 00:33:09+00:00,BUY,229.15,229.15,0.65,148.95,229.15,80.20,0.00,True,1.00,2025-01-01 06:25:03+00:00,0xd40a0282dcc17d28c73203e3ea538431114a901928aaf64bf7c5021429259c21,1,True,0.00,0.00,-0.00,0.00,2025-01-08T00:00:00Z,Cavaliers vs. Lakers,"[Sports, NBA, Games]",Sports,40959390651531425935743766952118482746316220645668151461809801143476988026290,Cavaliers,80.20,148.95,0.00,0.00,0.00,0.00,148.95,True,0.00,0.00,0.00,0.00,0.00,0.54,0.54
1,0x4fc19a907012ef58717ee7ceb1f89057f71aec92,0x0f11f83fb8e412686efcb9430fcbeb326e91f24f4f90539070a3912cf6c31724,40959390651531425935743766952118482746316220645668151461809801143476988026290,2025-01-01 00:46:17+00:00,BUY,999.98,770.83,0.65,501.04,770.83,269.79,4.86,True,1.00,2025-01-01 06:25:03+00:00,0x3fe549772f4a06e41961dce76fd154beeaa451316be33544969df7df4442ed3e,1,True,13.89,13.89,5.00,1.00,2025-01-08T00:00:00Z,Cavaliers vs. Lakers,"[Sports, NBA, Games]",Sports,40959390651531425935743766952118482746316220645668151461809801143476988026290,Cavaliers,269.79,501.04,0.00,0.00,0.00,0.00,649.99,True,0.25,0.09,0.25,0.09,9.03,0.54,0.54
2,0x8f19df5088b81721175218d225443ec28e55b0ca,0x0f11f83fb8e412686efcb9430fcbeb326e91f24f4f90539070a3912cf6c31724,27602378495287162690998628678069982978507358252069914020062082986279730001183,2025-01-01 04:13:25+00:00,BUY,77.08,77.08,0.01,0.77,0.00,-0.77,-0.00,False,0.00,2025-01-01 06:25:03+00:00,0xa338191eaf20b1ffce01be27f3c5752528de3d917a5b08c2e6a76a454438c68e,1,True,0.00,0.00,0.00,0.00,2025-01-08T00:00:00Z,Cavaliers vs. Lakers,"[Sports, NBA, Games]",Sports,40959390651531425935743766952118482746316220645668151461809801143476988026290,Lakers,-0.77,0.77,324.72,1.00,-0.77,-0.00,0.77,True,0.00,-0.00,0.00,-0.00,0.00,-1.00,NaN
3,0x8f19df5088b81721175218d225443ec28e55b0ca,0x0f11f83fb8e412686efcb9430fcbeb326e91f24f4f90539070a3912cf6c31724,27602378495287162690998628678069982978507358252069914020062082986279730001183,2025-01-01 04:13:27+00:00,BUY,85.49,8.41,0.01,0.08,0.00,-0.08,-0.00,False,0.00,2025-01-01 06:25:03+00:00,0xd46e4a50bf68998ad27d2e81e9b4c9de7c5859b32f4d8cdf5ed1f8a9e2eb5bc4,1,True,0.00,0.00,0.00,0.00,2025-01-08T00:00:00Z,Cavaliers vs. Lakers,"[Sports, NBA, Games]",Sports,40959390651531425935743766952118482746316220645668151461809801143476988026290,Lakers,-0.08,0.08,324.72,1.00,-0.08,-0.00,0.85,True,0.00,-0.00,0.00,-0.00,0.00,-1.00,NaN
4,0x8f19df5088b81721175218d225443ec28e55b0ca,0x0f11f83fb8e412686efcb9430fcbeb326e91f24f4f90539070a3912cf6c31724,27602378495287162690998628678069982978507358252069914020062082986279730001183,2025-01-01 04:13:33+00:00,BUY,999.99,914.50,0.01,9.14,0.00,-9.14,-0.00,False,0.00,2025-01-01 06:25:03+00:00,0xbd840f503fd918811305cf448bb75c77cbcbc7593cc5dc70db5964737f60f953,1,True,0.00,0.00,0.00,0.00,2025-01-08T00:00:00Z,Cavaliers vs. Lakers,"[Sports, NBA, Games]",Sports,40959390651531425935743766952118482746316220645668151461809801143476988026290,Lakers,-9.14,9.14,324.72,1.00,-9.14,-0.00,10.00,True,0.00,-0.00,0.00,-0.00,0.00,-1.00,NaN


In [63]:
wallet_vol_test, _ = compute_wallet_metrics(df_full[~df_full["is_train"]].copy())

/Users/vobornij/projects/polymarket/src/polymarket_analysis/wallet_selection/volatility.py:190: FutureWarning:

The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior



In [64]:
df_full['bucket'] = df_full['dt'].dt.round("5min")

In [65]:
# # WALLET COMBINATION CELL

# df = df_full[
#     (df_full['wallet'].isin(wallet_cohorts['predicting_group']['wallet']) | df_full['wallet'].isin(wallet_cohorts['copyable_group']['wallet']) | df_full['wallet'].isin(WALLET_FILTER))
#      & (df_full["is_train"])].copy()

# df['is_predicting_group'] = df['wallet'].isin(wallet_cohorts['predicting_group']['wallet'])
# df['is_filter_group'] = df['wallet'].isin(WALLET_FILTER)
# df['is_copyable_group'] = df['wallet'].isin(wallet_cohorts['copyable_group']['wallet']) & ~df['is_filter_group']



# print(f'predicting_size: {len(wallet_cohorts["predicting_group"]["wallet"])}')
# print(f'copyable_size: {len(wallet_cohorts["copyable_group"]["wallet"])}')

# print('\n')
# print('*** Total PnL / Copyable PnL before filtering ***')
# print(df[['pnl', 'copyable_pnl']].sum())
# print(f'wallet_roi={df["pnl"].sum() / df["notional"].sum()}')
# print(f'copyable_roi={df["copyable_pnl"].sum() / df["copyable_notional"].sum()}')


# print('\n')
# print('*** Predicting group PnL ***')
# print(df[df['is_predicting_group']][['pnl', 'copyable_pnl']].sum())
# print(f'wallet_roi={df[df["is_predicting_group"]]["pnl"].sum() / df[df["is_predicting_group"]]["notional"].sum()}')
# print(f'copyable_roi={df[df["is_predicting_group"]]["copyable_pnl"].sum() / df[df["is_predicting_group"]]["copyable_notional"].sum()}')

# print('\n')
# print('*** Copyable group PnL ***')
# print(df[df['is_copyable_group']][['pnl', 'copyable_pnl']].sum())
# print(f'wallet_roi={df[df["is_copyable_group"]]["pnl"].sum() / df[df["is_copyable_group"]]["notional"].sum()}')
# print(f'copyable_roi={df[df["is_copyable_group"]]["copyable_pnl"].sum() / df[df["is_copyable_group"]]["copyable_notional"].sum()}')

# df = df.groupby(['bucket', 'token_id', 'side']).agg(
#     copyable_pnl=('copyable_pnl', 'sum'),
#     wallet_count=('wallet', 'nunique'),
#     pnl=('pnl', 'sum'),
#     predicting_group_count=('is_predicting_group', 'sum'),
#     copyable_group_count=('is_copyable_group', 'sum'),
#     wallet_in_filter_count=('is_filter_group', 'sum'),
#     notional=('notional', 'sum'),
#     copyable_notional=('copyable_notional', 'sum'),
# )
# df.reset_index(inplace=True)
# df = df[
#     ((df['wallet_count'] > 1)
#     & (df['predicting_group_count'] <= 0)
#     & (df['copyable_group_count'] >= 0)
#      & (df['wallet_in_filter_count'] >= 0) )
#       | ( df['wallet_count'] == df['wallet_in_filter_count'])
#      ]

# print('\n')
# print('*** Combined PnL ***')
# print(df[['pnl', 'copyable_pnl']].sum())
# print(f'wallet_roi={df["pnl"].sum() / df["notional"].sum() if "notional" in df.columns else np.nan}')
# print(f'copyable_roi={df["copyable_pnl"].sum() / df["copyable_notional"].sum() if "copyable_notional" in df.columns else np.nan}')
# print(f'copyable_roi={df["copyable_pnl"].sum() / df["copyable_notional"].sum() if "copyable_notional" in df.columns else np.nan}')

In [66]:
for _wv in (wallet_vol_train, wallet_vol_train_full):
    _wv['copyable_pnl_factor'] = np.clip(
        _wv['copyable_pnl'] / _wv['total_pnl'].replace(0, np.nan),
        0,
        1.0,
    ).fillna(0.0)
    _wv['copyable_roi'] = _wv['average_roi'] * _wv['copyable_pnl_factor']
    _wv['buy_copyable_roi'] = _wv['buy_copyable_pnl'] / _wv['buy_copyable_notional'].replace(0, np.nan)

# Keep sorting deterministic for downstream filtering/inspection.
wallet_vol_train.sort_values('copyable_roi', ascending=False, inplace=True)
wallet_vol_train.reset_index(drop=True, inplace=True)
wallet_vol_train_full.sort_values('copyable_roi', ascending=False, inplace=True)
wallet_vol_train_full.reset_index(drop=True, inplace=True)


In [67]:
wallet_cohorts = {}

In [68]:
# Final wallet selection: threshold-based copyable + predicting groups
# COPYABLE_SCORE_MIN = -1
# PREDICTING_SCORE_MIN = -2

candidates = wallet_vol_train.copy()
candidates['buy_copyable_factor'] = candidates['buy_copyable_pnl'] / candidates['buy_pnl']

# Additional filter: keep only wallets active in the test period (> 0 test trades)
test_active_wallets = set(df_full.loc[~df_full['is_train'], 'wallet'].unique())
n_candidates = len(candidates)
candidates = candidates[candidates['wallet'].isin(test_active_wallets)].copy()
print(f"Test-active filter: kept {len(candidates):,} of {n_candidates:,} candidates")

for col, default in {
    'top10_pnl_pct': np.nan,
    'top_market_abs_pnl_pct': np.nan,
    'market_pnl_hhi': np.nan,
    'positive_bucket_share': np.nan,
}.items():
    if col not in candidates.columns:
        candidates[col] = default

for col in [
    'average_roi', 'median_roi', 'num_buckets', 'num_markets',
    'pnl_volatility', 'max_drawdown_to_pnl',
    'top_market_pnl_pct', 'top_market_abs_pnl_pct', 'top5_pnl_pct', 'top10_pnl_pct', 'market_pnl_hhi',
    'copyable_roi', 'copyable_pnl_factor', 'copyable_pnl', 'total_notional',
    'max_copyable_drawdown_to_copyable_pnl', 'worst5_pnl_pct', 'positive_bucket_share',
]:
    if col in candidates.columns:
        candidates[col] = pd.to_numeric(candidates[col], errors='coerce')

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "wallet_selection"))

from signal_lab.filters import (
    select_strategy_eligible,
    select_strategy_selection,
)

if "Esports" in TAGS or "Sports" in TAGS:
    variant = "sports"
elif "Politics" in TAGS:
    variant = "politics"
else:
    variant = "weather"

# Two separate stat frames passed to the filter functions: the full
# (non-filtered) stats drive the mask rules for now; the carved stats are
# available as the second parameter for rule-level adjustments.
eligible_wallets = select_strategy_eligible(
    wallet_vol_train_full, wallet_vol_train, variant=variant
)
copyable_wallets = select_strategy_selection(
    wallet_vol_train_full, wallet_vol_train, variant=variant
)

eligible_base = candidates[candidates['wallet'].isin(eligible_wallets)].copy()
if eligible_base.empty:
    raise ValueError('No wallets passed base eligibility filters.')

copyable_candidates = eligible_base[eligible_base['wallet'].isin(copyable_wallets)].copy()
copyable_candidates['copyable_efficiency'] = copyable_candidates['copyable_pnl'].fillna(0.0) / (copyable_candidates['total_notional'].fillna(0.0) + 1.0)
copyable_candidates['copyable_dd_ratio'] = (
    copyable_candidates['max_copyable_drawdown_to_copyable_pnl']
    .fillna(copyable_candidates['max_drawdown_to_pnl'])
    .fillna(0.0)
)


wallet_cohorts['copyable_group'] = (
    copyable_candidates
    .reset_index(drop=True)
)

predicting_pool = eligible_base[
    ~eligible_base['wallet'].isin(wallet_cohorts['copyable_group']['wallet'])
].copy()
wallet_cohorts['predicting_group'] = (
    predicting_pool
    .reset_index(drop=True)
)

overlap = set(wallet_cohorts['copyable_group']['wallet']) & set(wallet_cohorts['predicting_group']['wallet'])
if overlap:
    raise ValueError(f'Wallet groups must be disjoint; found overlap of {len(overlap)} wallets.')

wallet_cohorts['multi_filter'] = wallet_cohorts['copyable_group'].copy()

# Split the copyable group into 3 terciles by estimated_copyable_buy_sharpe.
# qcut raises when the values cannot form 3 unique bins (e.g. all-NaN sharpe
# after heavy filtering); skip the split and keep empty tercile cohorts so
# the summary print below still works.
_sharpe = wallet_cohorts['copyable_group']['estimated_copyable_buy_sharpe']
_SHARPE_LABELS = ['copyable_buy_sharpe_low', 'copyable_buy_sharpe_mid', 'copyable_buy_sharpe_high']
try:
    _terciles = pd.qcut(_sharpe, 3, labels=_SHARPE_LABELS)
except ValueError as _err:
    _terciles = None
    print(f"Sharpe terciles skipped: {_err}")
    print(
        f"  estimated_copyable_buy_sharpe: {_sharpe.notna().sum()}/{len(_sharpe)} valid, "
        f"{_sharpe.dropna().nunique()} distinct values"
    )
for _label in _SHARPE_LABELS:
    wallet_cohorts[_label] = (
        wallet_cohorts['copyable_group'][_terciles == _label].copy().reset_index(drop=True)
        if _terciles is not None
        else wallet_cohorts['copyable_group'].iloc[0:0].copy()
    )


print(f"Base-eligible wallets: {len(eligible_base):,}")
print(f"Copyable candidates: {len(copyable_candidates):,}")
print(f"Selected wallets (copyable group): {len(wallet_cohorts['copyable_group']):,} total copyable pnl {wallet_cohorts['copyable_group']['copyable_pnl'].sum()}")
print(f"Selected wallets (predicting group): {len(wallet_cohorts['predicting_group']):,} total predicting group copyable pnl {wallet_cohorts['predicting_group']['copyable_pnl'].sum()}")
print("Copyable sharpe terciles: " + ", ".join(f"{k}={len(wallet_cohorts[k])}" for k in ['copyable_buy_sharpe_low', 'copyable_buy_sharpe_mid', 'copyable_buy_sharpe_high']))
wallet_cohorts['copyable_group'].sort_values('estimated_copyable_buy_sharpe', key=abs, ascending=False).head(100)



Test-active filter: kept 1,165 of 2,495 candidates
Base-eligible wallets: 39
Copyable candidates: 25
Selected wallets (copyable group): 25 total copyable pnl 65653.658027786
Selected wallets (predicting group): 14 total predicting group copyable pnl 83369.99940797323
Copyable sharpe terciles: copyable_buy_sharpe_low=8, copyable_buy_sharpe_mid=8, copyable_buy_sharpe_high=9


,wallet,num_buckets,trade_count,total_notional,total_pnl,copyable_pnl,num_markets,median_dt,median_roi,average_roi,max_drawdown,max_copyable_drawdown,buy_pnl,buy_quantity,sell_pnl,sell_quantity,buy_notional,sell_notional,buy_copyable_quantity,buy_copyable_pnl,sell_copyable_quantity,sell_copyable_pnl,buy_roi,sell_roi,buy_copyable_notional,top5_pnl_pct,top10_pnl_pct,worst5_pnl_pct,positive_bucket_share,top_market_pnl_pct,top_market_abs_pnl_pct,market_pnl_hhi,pnl_volatility,estimated_sharpe,estimated_buy_sharpe,estimated_copyable_buy_sharpe,copyable_pnl_similarity,copyable_pnl_similarity_profit,copyable_pnl_similarity_loss,copyable_pnl_similarity_20m_100,copyable_pnl_similarity_profit_20m_100,copyable_pnl_similarity_loss_20m_100,max_drawdown_to_pnl,max_copyable_drawdown_to_copyable_pnl,return,max_drawdown_all,total_pnl_all,max_drawdown_all_to_pnl_all,copyable_pnl_factor,copyable_roi,buy_copyable_roi,buy_copyable_factor,copyable_efficiency,copyable_dd_ratio
20,0x0bcde02e0d053ed30532b77983a7e9bceffe811b,3,7,54722.86,3780.25,1730.61,3,2026-07-28 04:45:00+00:00,0.09,0.08,0.00,0.00,3780.25,58503.11,0.00,0.00,54722.86,0.00,24812.95,1730.61,0.00,0.00,0.07,0.00,23209.63,1.00,1.00,1.00,1.00,0.40,0.40,0.34,3.19,65.45,65.45,26.49,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.00,0.07,53.52,3706.77,0.01,0.46,0.03,0.07,0.46,0.03,0.00
6,0x46529b524aade9533ed5f9c83e51ad6dab446343,12,15,6299.08,3356.72,1164.80,9,2026-07-19 21:37:30+00:00,0.58,0.54,0.00,0.00,3356.72,9655.81,0.00,0.00,6299.08,0.00,3888.18,1164.80,0.00,0.00,0.53,0.00,2536.50,0.88,1.00,0.12,1.00,0.44,0.44,0.25,9.46,10.56,10.56,11.61,0.40,0.40,NaN,NaN,NaN,NaN,0.00,0.00,0.53,74.90,3202.85,0.02,0.35,0.19,0.46,0.35,0.18,0.00
3,0x55c137e7bb089e172b41e5420d24f303ce696485,7,7,5829.21,3956.70,1657.00,5,2026-06-27 18:40:00+00:00,0.54,0.67,0.00,0.00,3956.70,9785.91,0.00,0.00,5829.21,0.00,4046.44,1657.00,0.00,0.00,0.68,0.00,2410.36,0.91,1.00,0.73,1.00,0.29,0.29,0.22,4.46,12.61,12.61,9.76,0.59,0.59,NaN,NaN,NaN,NaN,0.00,0.00,0.68,214.99,3741.70,0.06,0.42,0.28,0.69,0.42,0.28,0.00
16,0x3a01dc75a904043f93f89bce183b45846ed5bc5e,5,5,10878.85,13184.21,1470.55,3,2026-07-27 11:30:00+00:00,0.47,0.71,0.00,0.00,13184.21,24063.06,0.00,0.00,10878.85,0.00,5647.32,1470.55,0.00,0.00,1.21,0.00,2553.14,1.00,1.00,1.00,1.00,0.88,0.88,0.78,46.53,7.18,7.18,9.62,0.14,0.14,NaN,NaN,NaN,NaN,0.00,0.00,1.21,423.93,12760.28,0.03,0.11,0.08,0.58,0.11,0.14,0.00
22,0xffe290a74a57db36b06cc88692b16f1cda322957,12,17,22029.48,12758.91,705.99,7,2026-06-22 12:55:00+00:00,0.64,0.39,1241.26,79.90,12758.91,37222.24,0.00,0.00,22029.48,0.00,2085.46,705.99,0.00,0.00,0.58,0.00,1234.25,0.74,1.00,0.26,0.90,0.23,0.19,0.21,7.28,8.86,8.86,7.39,0.78,0.76,NaN,NaN,NaN,NaN,0.10,0.11,0.58,2528.33,8788.53,0.29,0.06,0.02,0.57,0.06,0.03,0.11
2,0x9c4fe54fac3952bbced1da8b3c5301fe6fc20b9e,4,4,11744.38,2175.79,1432.74,3,2026-06-26 06:12:30+00:00,0.44,0.45,0.00,0.00,2175.79,13920.17,0.00,0.00,11744.38,0.00,7165.24,1432.74,0.00,0.00,0.19,0.00,6045.28,1.00,1.00,1.00,1.00,0.53,0.53,0.42,4.73,9.16,9.16,6.79,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.00,0.19,0.00,2175.79,0.00,0.66,0.30,0.24,0.66,0.12,0.00
7,0x7accd72259f3582ea4805386d09b311faed5c5b9,22,27,29142.12,9147.84,6742.01,21,2026-07-04 13:32:30+00:00,0.36,0.25,970.86,970.86,9147.84,41625.51,0.00,0.00,29142.12,0.00,30846.27,6742.01,0.00,0.00,0.31,0.00,21595.55,0.59,0.96,-0.22,0.86,0.15,0.10,0.11,5.35,7.75,7.75,6.72,0.93,0.92,NaN,NaN,NaN,NaN,0.11,0.14,0.31,2215.37,7352.38,0.30,0.74,0.18,0.31,0.74,0.23,0.14
13,0xbe506bbceb52a5cd0c5feeaf6a458e8727deb05c,4,4,12566.50,2036.90,1442.95,2,2026-07-10 13:52:30+00:00,0.19,0.16,0.00,0.00,2036.90,14603.40,0.00,0.00,12566.50,0.00,9018.43,1442.95,0.00,0.00,0.16,0.00,7760.53,1.00,1.00,1.00,1.00,0.86,0.86,0.76,12.46,7.46,7.46,6.37,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.00,0.16,0.00,2036.90,0.00,0.71,0.12,0.19,0.71,0.11,0.00
9,0x70a6798edbf21eb25cbb749569796d1e641aa2eb,39,72,5523.86,4026.90,1262.25,6,2026-07-19 12:35:00+00:00,0.67,0.56,310.57,0.46,4026.90,10225.92,0.00,0.00,5523.86,0.00,2776.58,1262.25,0.00,0

## Interactive wallet filter explorer

Fine-tune the selection in the browser: every numeric stat of both stat frames
(`wallet_vol_train_full` → `__full`, `wallet_vol_train` → `__carved`) gets a
dual range slider, pre-set to the `variant` thresholds. Click a wallet for its
train/test PnL panels, or switch to **picked aggregate** to see the cumulative
PnL of all lassoed wallets summed together; toggle the PnL variants
(wallet / copyable / capped, each for the **full** pre-carve stream and the
**carved** execution stream), the exposure lines and the extra carved
fractions.
Lasso to pick groups, then **Save selection** — the second cell reloads that
file and re-derives the passing wallets as a DataFrame.


In [69]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

from wallet_filter_explorer import open_wallet_filter_explorer

explorer = open_wallet_filter_explorer(
    wallet_vol_train_full,
    wallet_vol_train,
    variant=variant,
    eligible_wallets=eligible_wallets,
    selected_wallets=copyable_wallets,
    fills=df_full_buy_full,
    split_date=TRAIN_END,
    exposure_fracs=tuple(f for f in EXPOSURE_FRACS if f != EXPOSURE_FILTER_FRAC),
    out_dir=WORKSPACE_DIR,
)


Wallet filter explorer (sports) running at http://127.0.0.1:50508/
  save from the page, then re-run the load cell; stop with explorer.stop()


In [70]:
import importlib

import wallet_filter_explorer as _wfe

importlib.reload(_wfe)  # pick up module edits without restarting the kernel

selection = _wfe.load_selection(explorer.selection_path)
if selection is None:
    print(f"Nothing saved yet — use 'Save selection' in {explorer.url}")
else:
    explored_wallets = _wfe.apply_selection_filters(
        selection,
        wallet_vol_train_full,
        wallet_vol_train,
        fills=df_full_buy_full,
    )
    print(
        f"passing={len(explored_wallets)} "
        f"picked={int(explored_wallets['picked'].sum())} "
        f"copyable_pnl={explored_wallets['copyable_pnl__carved'].sum():,.0f}"
    )
    display(explored_wallets)

passing=40 picked=5 copyable_pnl=334,322


,wallet,num_buckets__full,trade_count__full,total_notional__full,total_pnl__full,copyable_pnl__full,num_markets__full,median_roi__full,average_roi__full,max_drawdown__full,max_copyable_drawdown__full,buy_pnl__full,buy_quantity__full,sell_pnl__full,sell_quantity__full,buy_notional__full,sell_notional__full,buy_copyable_quantity__full,buy_copyable_pnl__full,sell_copyable_quantity__full,sell_copyable_pnl__full,buy_roi__full,sell_roi__full,buy_copyable_notional__full,top5_pnl_pct__full,top10_pnl_pct__full,worst5_pnl_pct__full,positive_bucket_share__full,top_market_pnl_pct__full,top_market_abs_pnl_pct__full,market_pnl_hhi__full,pnl_volatility__full,estimated_sharpe__full,estimated_buy_sharpe__full,estimated_copyable_buy_sharpe__full,copyable_pnl_similarity__full,copyable_pnl_similarity_profit__full,copyable_pnl_similarity_loss__full,copyable_pnl_similarity_20m_100__full,copyable_pnl_similarity_profit_20m_100__full,copyable_pnl_similarity_loss_20m_100__full,max_drawdown_to_pnl__full,max_copyable_drawdown_to_copyable_pnl__full,return__full,max_drawdown_all__full,total_pnl_all__full,max_drawdown_all_to_pnl_all__full,copyable_pnl_factor__full,copyable_roi__full,buy_copyable_roi__full,median_dt_days_ago__full,num_buckets__carved,trade_count__carved,total_notional__carved,total_pnl__carved,copyable_pnl__carved,num_markets__carved,median_roi__carved,average_roi__carved,max_drawdown__carved,max_copyable_drawdown__carved,buy_pnl__carved,buy_quantity__carved,sell_pnl__carved,sell_quantity__carved,buy_notional__carved,sell_notional__carved,buy_copyable_quantity__carved,buy_copyable_pnl__carved,sell_copyable_quantity__carved,sell_copyable_pnl__carved,buy_roi__carved,sell_roi__carved,buy_copyable_notional__carved,top5_pnl_pct__carved,top10_pnl_pct__carved,worst5_pnl_pct__carved,positive_bucket_share__carved,top_market_pnl_pct__carved,top_market_abs_pnl_pct__carved,market_pnl_hhi__carved,pnl_volatility__carved,estimated_sharpe__carved,estimated_buy_sharpe__carved,estimated_copyable_buy_sharpe__carved,copyable_pnl_similarity__carved,copyable_pnl_similarity_profit__carved,copyable_pnl_similarity_loss__carved,copyable_pnl_similarity_20m_100__carved,copyable_pnl_similarity_profit_20m_100__carved,copyable_pnl_similarity_loss_20m_100__carved,max_drawdown_to_pnl__carved,max_copyable_drawdown_to_copyable_pnl__carved,return__carved,max_drawdown_all__carved,total_pnl_all__carved,max_drawdown_all_to_pnl_all__carved,copyable_pnl_factor__carved,copyable_roi__carved,buy_copyable_roi__carved,median_dt_days_ago__carved,picked
0,0x0bcde02e0d053ed30532b77983a7e9bceffe811b,62,342,294785.31,18434.33,1587.01,49,0.08,-0.10,1339.99,1401.66,18434.33,318729.91,0.00,0.00,294785.31,0.00,80187.71,1587.01,0.00,0.00,0.06,0.00,74163.60,0.59,0.93,-0.23,0.81,0.13,0.09,0.10,6.26,16.56,16.56,2.76,0.64,0.59,1.00,NaN,NaN,NaN,0.07,0.88,0.06,1499.95,14564.13,0.10,0.09,-0.01,0.02,94.37,3,7,54722.86,3780.25,1730.61,3,0.09,0.08,0.00,0.00,3780.25,58503.11,0.00,0.00,54722.86,0.00,24812.95,1730.61,0.00,0.00,0.07,0.00,23209.63,1.00,1.00,1.00,1.00,0.40,0.40,0.34,3.19,65.45,65.45,26.49,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.00,0.07,53.52,3706.77,0.01,0.46,0.03,0.07,94.34,True
1,0x0d18e30e252008d83f51dae00d36873ddd9e24a6,428,3196,2467016.10,471606.85,143712.79,51,0.14,0.03,57992.76,5954.57,471606.85,3329791.66,0.00,0.00,2467016.10,0.00,693286.02,143712.79,0.00,0.00,0.19,0.00,513650.08,0.50,0.79,-0.27,0.79,0.13,0.07,0.11,29.85,11.02,11.02,11.20,0.63,0.67,0.46,NaN,NaN,NaN,0.12,0.04,0.19,57992.76,453379.56,0.13,0.30,0.01,0.28,110.64,4,33,397502.81,95980.33,22384.83,4,0.20,0.20,0.00,0.00,95980.33,493483.14,0.00,0.00,397502.81,0.00,102044.77,22384.83,0.00,0.00,0.24,0.00,82197.51,1.00,1.00,1.00,1.00,0.53,0.53,0.43,55.41,10.68,10.68,8.33,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.00,0.24,0.00,95980.33,0.00,0.23,0.05,0.27,112.27,False
2,0x13e4bf768f4c2005af84d9f23c670ac0bc788b89,1118,3529,182448.62,29216.89,16291.27,624,0.16,0.20,3866.56,3762.60,29216.89,434147.55,0.00,0.00,182448.62,0.00,260803.06,16291.27,0.00,0.00,0.1

In [71]:
wallet_vol_train[wallet_vol_train['wallet'] == '0xc1d971eadb83e32358b828cef7a6f722770c5333']

,wallet,num_buckets,trade_count,total_notional,total_pnl,copyable_pnl,num_markets,median_dt,median_roi,average_roi,max_drawdown,max_copyable_drawdown,buy_pnl,buy_quantity,sell_pnl,sell_quantity,buy_notional,sell_notional,buy_copyable_quantity,buy_copyable_pnl,sell_copyable_quantity,sell_copyable_pnl,buy_roi,sell_roi,buy_copyable_notional,top5_pnl_pct,top10_pnl_pct,worst5_pnl_pct,positive_bucket_share,top_market_pnl_pct,top_market_abs_pnl_pct,market_pnl_hhi,pnl_volatility,estimated_sharpe,estimated_buy_sharpe,estimated_copyable_buy_sharpe,copyable_pnl_similarity,copyable_pnl_similarity_profit,copyable_pnl_similarity_loss,copyable_pnl_similarity_20m_100,copyable_pnl_similarity_profit_20m_100,copyable_pnl_similarity_loss_20m_100,max_drawdown_to_pnl,max_copyable_drawdown_to_copyable_pnl,return,max_drawdown_all,total_pnl_all,max_drawdown_all_to_pnl_all,copyable_pnl_factor,copyable_roi,buy_copyable_roi
1064,0xc1d971eadb83e32358b828cef7a6f722770c5333,61,158,57007.14,15567.50,5991.64,42,2026-07-22 12:10:00+00:00,0.47,0.42,4223.18,5903.78,15567.50,89811.50,0.00,0.00,57007.14,0.00,67768.34,5991.64,0.00,0.00,0.27,0.00,43015.42,0.53,0.87,-0.68,0.84,0.18,0.08,0.15,10.62,5.57,5.57,2.65,0.92,0.84,1.00,NaN,NaN,NaN,0.27,0.99,0.27,3700.99,16800.70,0.22,0.38,0.16,0.14


In [72]:
df_full.head()

,wallet,condition_id,token_id,dt,side,position,quantity,price,usdc_amount,final_value_usdc,trade_pnl,copyable_pnl,token_winner,final_price,last_condition_trade_ts,tx_hash,num_fills,is_train,copyable_qty_5m_100,avail_copy_qty_5m_100,avail_copy_total_vol_5m_100,avail_copy_count_5m_100,end_date_iso,question,tags,primary_tag,winner_token_id,outcome,pnl,notional,deployed_capital_limit,deployed_capital_frac,pnl_limited,copyable_pnl_limited,deployed_capital,is_carved,copyable_qty_2x,copyable_pnl_2x,copyable_qty_0.5x,copyable_pnl_0.5x,copyable_notional,roi,copyable_roi,bucket
0,0x4fc19a907012ef58717ee7ceb1f89057f71aec92,0x0f11f83fb8e412686efcb9430fcbeb326e91f24f4f90539070a3912cf6c31724,40959390651531425935743766952118482746316220645668151461809801143476988026290,2025-01-01 00:33:09+00:00,BUY,229.15,229.15,0.65,148.95,229.15,80.20,0.00,True,1.00,2025-01-01 06:25:03+00:00,0xd40a0282dcc17d28c73203e3ea538431114a901928aaf64bf7c5021429259c21,1,True,0.00,0.00,-0.00,0.00,2025-01-08T00:00:00Z,Cavaliers vs. Lakers,"[Sports, NBA, Games]",Sports,40959390651531425935743766952118482746316220645668151461809801143476988026290,Cavaliers,80.20,148.95,0.00,0.00,0.00,0.00,148.95,True,0.00,0.00,0.00,0.00,0.00,0.54,0.54,2025-01-01 00:35:00+00:00
1,0x4fc19a907012ef58717ee7ceb1f89057f71aec92,0x0f11f83fb8e412686efcb9430fcbeb326e91f24f4f90539070a3912cf6c31724,40959390651531425935743766952118482746316220645668151461809801143476988026290,2025-01-01 00:46:17+00:00,BUY,999.98,770.83,0.65,501.04,770.83,269.79,4.86,True,1.00,2025-01-01 06:25:03+00:00,0x3fe549772f4a06e41961dce76fd154beeaa451316be33544969df7df4442ed3e,1,True,13.89,13.89,5.00,1.00,2025-01-08T00:00:00Z,Cavaliers vs. Lakers,"[Sports, NBA, Games]",Sports,40959390651531425935743766952118482746316220645668151461809801143476988026290,Cavaliers,269.79,501.04,0.00,0.00,0.00,0.00,649.99,True,0.25,0.09,0.25,0.09,9.03,0.54,0.54,2025-01-01 00:45:00+00:00
2,0x8f19df5088b81721175218d225443ec28e55b0ca,0x0f11f83fb8e412686efcb9430fcbeb326e91f24f4f90539070a3912cf6c31724,27602378495287162690998628678069982978507358252069914020062082986279730001183,2025-01-01 04:13:25+00:00,BUY,77.08,77.08,0.01,0.77,0.00,-0.77,-0.00,False,0.00,2025-01-01 06:25:03+00:00,0xa338191eaf20b1ffce01be27f3c5752528de3d917a5b08c2e6a76a454438c68e,1,True,0.00,0.00,0.00,0.00,2025-01-08T00:00:00Z,Cavaliers vs. Lakers,"[Sports, NBA, Games]",Sports,40959390651531425935743766952118482746316220645668151461809801143476988026290,Lakers,-0.77,0.77,324.72,1.00,-0.77,-0.00,0.77,True,0.00,-0.00,0.00,-0.00,0.00,-1.00,NaN,2025-01-01 04:15:00+00:00
3,0x8f19df5088b81721175218d225443ec28e55b0ca,0x0f11f83fb8e412686efcb9430fcbeb326e91f24f4f90539070a3912cf6c31724,27602378495287162690998628678069982978507358252069914020062082986279730001183,2025-01-01 04:13:27+00:00,BUY,85.49,8.41,0.01,0.08,0.00,-0.08,-0.00,False,0.00,2025-01-01 06:25:03+00:00,0xd46e4a50bf68998ad27d2e81e9b4c9de7c5859b32f4d8cdf5ed1f8a9e2eb5bc4,1,True,0.00,0.00,0.00,0.00,2025-01-08T00:00:00Z,Cavaliers vs. Lakers,"[Sports, NBA, Games]",Sports,40959390651531425935743766952118482746316220645668151461809801143476988026290,Lakers,-0.08,0.08,324.72,1.00,-0.08,-0.00,0.85,True,0.00,-0.00,0.00,-0.00,0.00,-1.00,NaN,2025-01-01 04:15:00+00:00
4,0x8f19df5088b81721175218d225443ec28e55b0ca,0x0f11f83fb8e412686efcb9430fcbeb326e91f24f4f90539070a3912cf6c31724,27602378495287162690998628678069982978507358252069914020062082986279730001183,2025-01-01 04:13:33+00:00,BUY,999.99,914.50,0.01,9.14,0.00,-9.14,-0.00,False,0.00,2025-01-01 06:25:03+00:00,0xbd840f503fd918811305cf448bb75c77cbcbc7593cc5dc70db5964737f60f953,1,True,0.00,0.00,0.00,0.00,2025-01-08T00:00:00Z,Cavaliers vs. Lakers,"[Sports, NBA, Games]",Sports,40959390651531425935743766952118482746316220645668151461809801143476988026290,Lakers,-9.14,9.14,324.72,1.00,-9.14,-0.00,10.00,True,0.00,-0.00,0.00,-0.00,0.00,-1.00,NaN,2025-01-01 04:15:00+00:00


In [73]:
wallet_vol_train.query("wallet == '0xc3973025163de81e367ad29c21e296e51684bb3b'")

,wallet,num_buckets,trade_count,total_notional,total_pnl,copyable_pnl,num_markets,median_dt,median_roi,average_roi,max_drawdown,max_copyable_drawdown,buy_pnl,buy_quantity,sell_pnl,sell_quantity,buy_notional,sell_notional,buy_copyable_quantity,buy_copyable_pnl,sell_copyable_quantity,sell_copyable_pnl,buy_roi,sell_roi,buy_copyable_notional,top5_pnl_pct,top10_pnl_pct,worst5_pnl_pct,positive_bucket_share,top_market_pnl_pct,top_market_abs_pnl_pct,market_pnl_hhi,pnl_volatility,estimated_sharpe,estimated_buy_sharpe,estimated_copyable_buy_sharpe,copyable_pnl_similarity,copyable_pnl_similarity_profit,copyable_pnl_similarity_loss,copyable_pnl_similarity_20m_100,copyable_pnl_similarity_profit_20m_100,copyable_pnl_similarity_loss_20m_100,max_drawdown_to_pnl,max_copyable_drawdown_to_copyable_pnl,return,max_drawdown_all,total_pnl_all,max_drawdown_all_to_pnl_all,copyable_pnl_factor,copyable_roi,buy_copyable_roi
1735,0xc3973025163de81e367ad29c21e296e51684bb3b,452,528,155970.43,15981.68,3055.86,366,2026-05-20 05:52:30+00:00,0.23,0.16,3066.02,3426.01,15981.68,232845.40,0.00,0.00,155970.43,0.00,180890.62,3055.86,0.00,0.00,0.10,0.00,121168.76,0.33,0.55,-0.36,0.79,0.09,0.02,0.09,3.26,3.85,3.85,0.87,0.91,0.81,0.98,NaN,NaN,NaN,0.19,1.12,0.10,3732.65,12745.93,0.29,0.19,0.03,0.03


In [74]:
print("TEST")
wallet_vol_test[wallet_vol_test['wallet'] == '0xc3973025163de81e367ad29c21e296e51684bb3b']

TEST


,wallet,num_buckets,trade_count,total_notional,total_pnl,copyable_pnl,num_markets,median_dt,median_roi,average_roi,max_drawdown,max_copyable_drawdown,buy_pnl,buy_quantity,sell_pnl,sell_quantity,buy_notional,sell_notional,buy_copyable_quantity,buy_copyable_pnl,sell_copyable_quantity,sell_copyable_pnl,buy_roi,sell_roi,buy_copyable_notional,top5_pnl_pct,top10_pnl_pct,worst5_pnl_pct,positive_bucket_share,top_market_pnl_pct,top_market_abs_pnl_pct,market_pnl_hhi,pnl_volatility,estimated_sharpe,estimated_buy_sharpe,estimated_copyable_buy_sharpe,copyable_pnl_similarity,copyable_pnl_similarity_profit,copyable_pnl_similarity_loss,copyable_pnl_similarity_20m_100,copyable_pnl_similarity_profit_20m_100,copyable_pnl_similarity_loss_20m_100,max_drawdown_to_pnl,max_copyable_drawdown_to_copyable_pnl,return
53,0xc3973025163de81e367ad29c21e296e51684bb3b,1700,2452,1311841.56,87645.50,26139.10,1133,2026-08-31 11:27:30+00:00,0.16,0.05,21666.25,33089.92,87645.50,2528626.66,0.00,0.00,1311841.56,0.00,2122457.69,26139.10,0.00,0.00,0.07,0.00,1101122.70,0.38,0.64,-0.18,0.61,0.10,0.01,0.11,4.56,4.97,4.97,1.67,0.93,0.91,0.97,NaN,NaN,NaN,0.25,1.27,0.07


In [75]:
selected_wallets_vol = wallet_vol_test[wallet_vol_test['wallet'].isin(wallet_cohorts['predicting_group'].wallet)]
print("TEST")
selected_wallets_vol.sort_values(by='copyable_pnl', key=abs, ascending=False).head(50)

TEST


,wallet,num_buckets,trade_count,total_notional,total_pnl,copyable_pnl,num_markets,median_dt,median_roi,average_roi,max_drawdown,max_copyable_drawdown,buy_pnl,buy_quantity,sell_pnl,sell_quantity,buy_notional,sell_notional,buy_copyable_quantity,buy_copyable_pnl,sell_copyable_quantity,sell_copyable_pnl,buy_roi,sell_roi,buy_copyable_notional,top5_pnl_pct,top10_pnl_pct,worst5_pnl_pct,positive_bucket_share,top_market_pnl_pct,top_market_abs_pnl_pct,market_pnl_hhi,pnl_volatility,estimated_sharpe,estimated_buy_sharpe,estimated_copyable_buy_sharpe,copyable_pnl_similarity,copyable_pnl_similarity_profit,copyable_pnl_similarity_loss,copyable_pnl_similarity_20m_100,copyable_pnl_similarity_profit_20m_100,copyable_pnl_similarity_loss_20m_100,max_drawdown_to_pnl,max_copyable_drawdown_to_copyable_pnl,return
175,0x1f7105a18d9f36aef7c83e5df210e57cf487b2e4,49,974,2416767.83,263788.01,138553.58,42,2026-08-25 16:50:00+00:00,0.05,0.13,81852.93,38168.75,263788.01,2851928.95,0.00,0.00,2416767.83,0.00,745581.48,138553.58,0.00,0.00,0.11,0.00,631817.05,0.74,1.08,-0.39,0.93,0.22,0.13,0.18,36.47,5.56,5.56,4.79,0.76,0.84,NaN,NaN,NaN,NaN,0.31,0.28,0.11
123,0x6a937f25ab00ec1edb4416d46485e7aba7a98e36,290,369,664462.13,-32580.31,-35339.92,220,2026-08-13 13:22:30+00:00,0.08,-0.01,46281.31,48915.40,-32580.31,785999.87,0.00,0.00,664462.13,0.00,646488.34,-35339.92,0.00,0.00,-0.05,0.00,546523.01,-0.55,-0.86,1.08,0.85,-0.15,0.04,NaN,NaN,-4.63,-4.63,-5.29,0.98,0.98,0.98,NaN,NaN,NaN,NaN,NaN,-0.05
53,0xc3973025163de81e367ad29c21e296e51684bb3b,1700,2452,1311841.56,87645.50,26139.10,1133,2026-08-31 11:27:30+00:00,0.16,0.05,21666.25,33089.92,87645.50,2528626.66,0.00,0.00,1311841.56,0.00,2122457.69,26139.10,0.00,0.00,0.07,0.00,1101122.70,0.38,0.64,-0.18,0.61,0.10,0.01,0.11,4.56,4.97,4.97,1.67,0.93,0.91,0.97,NaN,NaN,NaN,0.25,1.27,0.07
54,0xc1d971eadb83e32358b828cef7a6f722770c5333,92,191,134819.96,27328.04,24643.86,75,2026-08-16 12:00:00+00:00,0.29,0.05,8522.79,8522.79,27328.04,217634.89,0.00,0.00,134819.96,0.00,199680.18,24643.86,0.00,0.00,0.20,0.00,123697.42,0.78,1.14,-0.51,0.68,0.30,0.10,0.22,11.25,4.62,4.62,4.39,0.99,0.99,0.99,NaN,NaN,NaN,0.31,0.35,0.20
132,0x26273b50e0dce6f2f17795f28ca7a9303517fa13,52,131,91732.59,2586.92,-7199.74,49,2026-08-28 13:17:30+00:00,-1.00,0.01,11634.12,11770.28,2586.92,191648.28,0.00,0.00,91732.59,0.00,114828.75,-7199.74,0.00,0.00,0.03,0.00,54962.87,7.97,13.74,-6.94,0.47,1.87,0.06,22.98,50.97,0.29,0.29,-1.29,0.76,0.72,0.83,NaN,NaN,NaN,4.50,NaN,0.03
595,0x3f176a240c7b6c0e470bf150f43fc1c883de7035,68,118,58987.35,12405.62,4350.13,45,2026-08-29 10:52:30+00:00,0.39,0.15,5475.43,2581.97,12405.62,89781.76,0.00,0.00,58987.35,0.00,46429.18,4350.13,0.00,0.00,0.21,0.00,30504.35,0.90,1.25,-0.65,0.74,0.52,0.17,0.41,14.34,2.96,2.96,2.24,0.60,0.52,0.79,NaN,NaN,NaN,0.44,0.59,0.21
43,0xcfe62595d0daadfc34e724b3b79dba176a734fad,46,74,38694.46,1023.21,4202.40,36,2026-08-09 07:40:00+00:00,0.27,-0.07,3744.12,2657.68,1023.21,61141.61,0.00,0.00,38694.46,0.00,41184.82,4202.40,0.00,0.00,0.03,0.00,26064.48,6.63,9.50,-6.31,0.60,1.62,0.10,9.73,33.88,1.12,1.12,6.04,0.89,0.97,0.80,NaN,NaN,NaN,3.66,0.63,0.03
488,0x7ea571c40408f340c1c8fc8eaacebab53c1bde7b,29,66,177026.38,11797.64,2760.28,12,2026-09-26 15:05:00+00:00,0.16,0.14,35054.38,35054.38,11797.64,279103.79,0.00,0.00,177026.38,0.00,171810.89,2760.28,0.00,0.00,0.07,0.00,108974.02,4.20,4.41,-3.44,0.89,2.24,0.38,6.43,180.09,0.54,0.54,0.13,0.98,0.97,NaN,NaN,NaN,NaN,2.97,12.70,0.07
251,0x73492e9ea5e7423cc0bb10a7169a0c81972dcc3a,15,73,13410.22,-3122.35,-2528.21,12,2026-08-26 20:00:00+00:00,-1.00,-0.34,3735.03,3019.59,-3122.35,24088.24,0.00,0.00,13410.22,0.00,17125.94,-2528.21,0.00,0.00,-0.23,0.00,9534.22,-1.12,-0.31,2.11,0.50,-0.56,0.20,NaN,NaN,-2.27,-2.27,-2.39,0.86,0.94,0.82,NaN,NaN,NaN,NaN,NaN,-0.23
198,0xa844c3d80fca2cb601efc9843ffe677e433a83b8,78,86,86703.53,-1148.94,-1469.51,63,2026-08-08 16:42:30+00:00,0.18,-0.06,11489.86,11705.96,-1148.94,127841.06,0.00,0.00,86703.53,0.00,121634.52,-1469.51,0.00,0.00,-0.01,0.00,82494.18,

In [102]:
# ── Individual wallet inspection: PnL / copyable PnL variants / exposure ─────
# Per wallet: two stacked subplots (train / test) with cumulative wallet PnL,
# copyable-PnL variants (raw/unlimited) and their copy exposure (secondary
# axis). Additionally the bankroll-discipline carve variants per
# EXPOSURE_FRACS: T_{w,f} is fit on the wallet's TRAIN fills via
# wallet_capture_thresholds (smallest deployed capital capturing fraction `f`
# of the wallet's train copyable PnL; f == 1 is the raw baseline above), and
# the same thresholds carve the fills in both subplots. The exposure-capped
# variants (pnl_limited / copyable_pnl_limited, per-wallet cap) are drawn
# raw and on the carved fill sets.
import plotly.colors
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from polymarket_analysis.visualization.wallet_plots import copyable_exposure_series
from polymarket_analysis.wallet_selection.exposure import wallet_capture_thresholds

INSPECT_WALLETS = [
    # "0xc1d971eadb83e32358b828cef7a6f722770c5333",
    # "0xc3973025163de81e367ad29c21e296e51684bb3b",
    # "0xf64745161e46cbd2143f021edd92cf6bed850ccb"
    "0x08c263965a9cbec5e83ac1d6f14b9bbc957bc023"
]

PNL_TO_QTY = {
    COPYABLE_PNL_BASE: "copyable_qty_5m_100",
    **{f"copyable_pnl_{k}": cfg["qty_col"] for k, cfg in COPYABLE_VARIANTS.items()},
}
LINE_COLORS = {
    "pnl": "#1f77b4",
    **{c: col for c, col in zip(COPYABLE_PNL_COLS, ["#ff7f0e", "#2ca02c", "#d62728", "#9467bd"])},
    **{c: col for c, col in zip(CAPPED_PNL_COLS, ["#8c564b", "#e377c2"])},
}

CARVE_FRACS = sorted({f for f in EXPOSURE_FRACS if f < 1}, reverse=True)
FRAC_COLORS = {f: col for f, col in zip(CARVE_FRACS, plotly.colors.qualitative.Dark2)}

for wallet in INSPECT_WALLETS:
    w = df_full[df_full["wallet"] == wallet].copy()
    if w.empty:
        print(f"{wallet}: no fills in df_full, skipping")
        continue

    # per-fill copy exposure (price × copyable qty)
    for pnl_col, qty_col in PNL_TO_QTY.items():
        w[f"{pnl_col}_exposure"] = np.where(
            w[qty_col] > 0,
            w["price"] * w[qty_col],
            np.nan,
        )
    w["deployed_capital"] = w["position"] * w["price"]

    # capped-variant exposures: capital actually deployed under the per-wallet cap
    w["pnl_limited_exposure"] = w["notional"] * w["deployed_capital_frac"]
    w["copyable_pnl_limited_exposure"] = np.where(
        w["copyable_qty_5m_100"] > 0,
        w["price"] * w["copyable_qty_5m_100"] * w["deployed_capital_frac"],
        np.nan,
    )

    # carve thresholds T_{w,f}: fit on the wallet's train fills only
    fit = w[w["is_train"] & w["deployed_capital"].notna()][
        ["wallet", EXPOSURE_PNL_COL, "deployed_capital"]
    ]
    capture = wallet_capture_thresholds(fit, EXPOSURE_FRACS, EXPOSURE_PNL_COL, "deployed_capital")
    carve_threshold = {}
    for frac in CARVE_FRACS:
        rows_ = capture[capture["frac"] == frac]
        if not rows_.empty:
            carve_threshold[frac] = float(rows_["threshold"].iloc[0])
    if len(carve_threshold) < len(CARVE_FRACS):
        print(f"{wallet}: no positive train {EXPOSURE_PNL_COL} — some/all carve variants skipped")

    fig = make_subplots(
        rows=2,
        cols=1,
        subplot_titles=("Train", "Test"),
        specs=[[{"secondary_y": True}], [{"secondary_y": True}]],
    )

    for row, (period, mask) in enumerate(
        [("train", w["is_train"]), ("test", ~w["is_train"])], start=1
    ):
        sub = w[mask].sort_values("dt")
        if sub.empty:
            continue

        fig.add_trace(
            go.Scatter(
                x=sub["dt"],
                y=sub["pnl"].cumsum(),
                mode="lines",
                name="pnl",
                legendgroup="pnl",
                showlegend=(row == 1),
                line=dict(color=LINE_COLORS["pnl"], width=2),
            ),
            row=row, col=1, secondary_y=False,
        )
        for c in COPYABLE_PNL_COLS:
            fig.add_trace(
                go.Scatter(
                    x=sub["dt"],
                    y=sub[c].cumsum(),
                    mode="lines",
                    name=c,
                    legendgroup=c,
                    showlegend=(row == 1),
                    line=dict(color=LINE_COLORS[c], width=2, dash="dash"),
                ),
                row=row, col=1, secondary_y=False,
            )
            exp = copyable_exposure_series(sub, f"{c}_exposure")
            if not exp.empty:
                fig.add_trace(
                    go.Scatter(
                        x=exp["dt"],
                        y=exp["exposure"],
                        mode="lines",
                        name=f"{c} exposure",
                        legendgroup=f"{c}_exp",
                        showlegend=(row == 1),
                        line=dict(color=LINE_COLORS[c], width=1.5, dash="dot"),
                        opacity=0.75,
                    ),
                    row=row, col=1, secondary_y=True,
                )

        # exposure-capped PnL (per-wallet deployed_capital_limit) + capped exposure
        for c in CAPPED_PNL_COLS:
            fig.add_trace(
                go.Scatter(
                    x=sub["dt"],
                    y=sub[c].cumsum(),
                    mode="lines",
                    name=f"{c} (capped)",
                    legendgroup=f"{c}_capped",
                    showlegend=(row == 1),
                    line=dict(color=LINE_COLORS[c], width=2, dash="longdash"),
                ),
                row=row, col=1, secondary_y=False,
            )
            exp = copyable_exposure_series(sub, f"{c}_exposure")
            if not exp.empty:
                fig.add_trace(
                    go.Scatter(
                        x=exp["dt"],
                        y=exp["exposure"],
                        mode="lines",
                        name=f"{c} exposure (capped)",
                        legendgroup=f"{c}_exp_capped",
                        showlegend=(row == 1),
                        line=dict(color=LINE_COLORS[c], width=1.5, dash="dot"),
                        opacity=0.75,
                    ),
                    row=row, col=1, secondary_y=True,
                )

        # carved copyable PnL + exposure per bankroll frac (train-fit T_{w,f})
        for frac in CARVE_FRACS:
            if frac not in carve_threshold:
                continue
            carved = sub[
                sub["deployed_capital"].notna()
                & (sub["deployed_capital"] >= carve_threshold[frac])
            ]
            if carved.empty:
                continue
            color = FRAC_COLORS[frac]
            for c in COPYABLE_PNL_COLS:
                group = f"{c} (frac {frac:g})"
                fig.add_trace(
                    go.Scatter(
                        x=carved["dt"],
                        y=carved[c].cumsum(),
                        mode="lines",
                        name=group,
                        legendgroup=f"{c}_frac{frac:g}",
                        showlegend=(row == 1),
                        line=dict(color=color, width=1.5, dash="dash"),
                        opacity=0.9,
                    ),
                    row=row, col=1, secondary_y=False,
                )
                exp = copyable_exposure_series(carved, f"{c}_exposure")
                if not exp.empty:
                    fig.add_trace(
                        go.Scatter(
                            x=exp["dt"],
                            y=exp["exposure"],
                            mode="lines",
                            name=f"{c} exposure (frac {frac:g})",
                            legendgroup=f"{c}_exp_frac{frac:g}",
                            showlegend=(row == 1),
                            line=dict(color=color, width=1.5, dash="dot"),
                            opacity=0.7,
                        ),
                        row=row, col=1, secondary_y=True,
                    )

            # capped PnL + capped exposure on the same carved fills
            for c in CAPPED_PNL_COLS:
                fig.add_trace(
                    go.Scatter(
                        x=carved["dt"],
                        y=carved[c].cumsum(),
                        mode="lines",
                        name=f"{c} (capped, frac {frac:g})",
                        legendgroup=f"{c}_capped_frac{frac:g}",
                        showlegend=(row == 1),
                        line=dict(color=FRAC_COLORS[frac], width=1.5, dash="longdash"),
                        opacity=0.9,
                    ),
                    row=row, col=1, secondary_y=False,
                )
                exp = copyable_exposure_series(carved, f"{c}_exposure")
                if not exp.empty:
                    fig.add_trace(
                        go.Scatter(
                            x=exp["dt"],
                            y=exp["exposure"],
                            mode="lines",
                            name=f"{c} exposure (capped, frac {frac:g})",
                            legendgroup=f"{c}_exp_capped_frac{frac:g}",
                            showlegend=(row == 1),
                            line=dict(color=FRAC_COLORS[frac], width=1.5, dash="dot"),
                            opacity=0.7,
                        ),
                        row=row, col=1, secondary_y=True,
                    )

    fig.update_layout(
        title=f"Wallet {wallet} — cumulative PnL, copyable PnL (variants + carve fracs) and exposure",
        template="plotly_white",
        height=800,
    )
    fig.update_yaxes(title_text="Cumulative PnL (USDC)", secondary_y=False)
    fig.update_yaxes(title_text="Exposure (USDC)", secondary_y=True)
    fig.show(renderer="browser")


In [77]:
df_full['is_copyable'] = df_full['avail_copy_count_5m_100'] >=1

In [78]:
df_full[df_full['wallet'].isin(wallet_cohorts['copyable_group']['wallet'])] \
    .groupby('wallet') \
    .agg(
        trade_count=('wallet', 'count'),
        copyable_count=('is_copyable', 'sum')
    ) \
    .assign(copyable_ratio=lambda x: x['copyable_count'] / x['trade_count']) \
    .reset_index().sort_values('trade_count', ascending=False).head(5)

,wallet,trade_count,copyable_count,copyable_ratio
15,0x880dbf161120207d47aef05ec774b331bcec39bc,391,362,0.93
7,0x46529b524aade9533ed5f9c83e51ad6dab446343,229,192,0.84
11,0x70a6798edbf21eb25cbb749569796d1e641aa2eb,214,123,0.57
1,0x13e4bf768f4c2005af84d9f23c670ac0bc788b89,210,159,0.76
22,0xc34b4901be324718c176100e350211d97b199296,210,169,0.80


In [79]:
if(WALLET_FILTER is not None):
    wallet_cohorts['copyable_group'] = wallet_cohorts['copyable_group'].query("wallet in @WALLET_FILTER").copy().reset_index(drop=True)

In [80]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)


In [81]:
filter_wallets = set(wallet_cohorts['multi_filter']["wallet"])
df_test =  df_full[
    (df_full['wallet'].isin(filter_wallets))
    ].groupby(['question', 'condition_id', 'outcome']).agg(
    # ].groupby(['wallet']).agg(
        pnl_sum=('pnl', 'sum'),
        trade_count=('pnl', 'size'),
        wallet_count=('wallet', 'nunique'),
    ).reset_index().sort_values('pnl_sum', key=abs, ascending=False)

big_conditions = df_test.groupby('condition_id').agg(
    abs_pnl_sum=('pnl_sum', lambda x: abs(x).sum()),
    min_abs_pnl_pct=('pnl_sum', lambda x: abs(x).min() / abs(x).sum() if abs(x).sum() > 0 else 0)
).reset_index().sort_values('abs_pnl_sum', ascending=False)



print(df_test['pnl_sum'].sum())
df_test = df_test[df_test['condition_id'].isin(big_conditions['condition_id'])].sort_values('question', ascending=False).merge(big_conditions, on='condition_id', how='left')

print(df_test[df_test['min_abs_pnl_pct']<0.1]['pnl_sum'].sum()) 

df_test[df_test['min_abs_pnl_pct'] < 0.1].sort_values(['abs_pnl_sum'], ascending=False).head(2)

104652.82096165301
-3113.115901


,question,condition_id,outcome,pnl_sum,trade_count,wallet_count,abs_pnl_sum,min_abs_pnl_pct
5,Will Wrexham AFC win on 2026-08-17?,0x850e66aaf30b1e45cb0ac1f6009e770901b05354be12e43dca16b698264f55b2,Yes,-1964.15,1,1,2046.00,0.04
6,Will Wrexham AFC win on 2026-08-17?,0x850e66aaf30b1e45cb0ac1f6009e770901b05354be12e43dca16b698264f55b2,No,81.85,3,1,2046.00,0.04


In [82]:
wallets = set(wallet_cohorts['copyable_group']['wallet'])

df = df_full #[~df_full["is_train"]].copy()
print(len(df))

df_wallets = df[
    (df['wallet'].isin(wallets))
    & ~df['is_train']
    ].copy()
print(len(df_wallets))

df = df_wallets.groupby('condition_id').agg(
    pnl=('pnl', 'sum'),
    notional=('notional', 'sum'),
    sell_count=('side', lambda x: (x == 'SELL').sum()),
    buy_count=('side', lambda x: (x == 'BUY').sum()),
    wallets=('wallet', lambda x: x.nunique()),
).sort_values(by="pnl", key=np.abs, ascending=False).merge(mdf, on="condition_id", how="left")

1617959
1371


In [83]:
wallet_set = set(wallet_cohorts['copyable_group']['wallet'])
print(f"Copyable wallets: {len(wallet_set)} with fills {df_wallets[df_wallets['wallet'].isin(wallet_set)].shape[0]} with total PnL: {df_wallets[df_wallets['wallet'].isin(wallet_set)]['pnl'].sum():,.2f}")
df_wallets[df_wallets['wallet'].isin(wallet_set)]['trade_pnl'].sum()

Copyable wallets: 25 with fills 1371 with total PnL: -29,694.33


np.float64(-29694.330809499996)

In [84]:
df_wallets[df_wallets['wallet'].isin(wallet_set)].groupby('wallet').agg(
    pnl=('trade_pnl', 'sum'),
    notional=('notional', 'sum'),
    trades=('wallet', 'count'),
    copyable_pnl=('copyable_pnl', 'sum'),
    copyable_qty_5m_100=('copyable_qty_5m_100', 'sum'),
    ).sort_values(by="pnl", key=np.abs, ascending=False).head(20)

,pnl,notional,trades,copyable_pnl,copyable_qty_5m_100
wallet,,,,,
0x0bcde02e0d053ed30532b77983a7e9bceffe811b,-24569.40,35730.85,16,-24569.40,42989.35
0x5c2311acccc5e9a172f0ce23d4480177bc00cab3,-21096.91,115708.69,155,-29310.59,92711.81
0xad2f5bf835fee93d6e7403a274cdc40f7a2dac6a,12366.43,248555.25,42,1224.09,12938.34
0x9c4fe54fac3952bbced1da8b3c5301fe6fc20b9e,11398.26,34775.26,5,2226.85,10225.62
0xf2cae58953c73cb722ba70ab694d3779b6e4b467,-10092.05,21200.00,4,-10341.38,21687.93
0xc34b4901be324718c176100e350211d97b199296,8337.61,233849.60,170,-9163.34,199118.00
0xc2494b32d65dea3a3a80712589d10bc5733f8a61,-6466.00,10466.00,4,-1708.35,4229.09
0x880dbf161120207d47aef05ec774b331bcec39bc,-5850.01,75339.18,322,-6542.12,82003.17
0x46529b524aade9533ed5f9c83e51ad6dab446343,-5201.27,89253.79,214,-14649.97,94118.70


In [85]:
# ── Copyable-group test stats: wallet vs copyable variants ──────────────────
# Cohort-level stats on the test period for the copyable_group: the wallets'
# own PnL (trade_pnl) and each copyable PnL variant. ROI uses the capital
# deployed (sum of per-fill exposure). Sharpe is the annualized daily-PnL
# Sharpe, zero-filled across the test date range. Exposure quantiles come
# from the capital-at-risk time profile (fills add exposure, released 1d
# after the contract's last trade).
#
# In addition to the full rows, each variant also gets a row per exposure
# carve (`EXPOSURE_FRACS`, frac < 1). The carve is a bankroll-discipline
# filter: per wallet keep the fills whose deployed capital
# (position * price, the post-trade USDC value of the book on that
# condition/outcome) is >= T_{w,f}, where T is fit on TRAIN via
# wallet_capture_thresholds (the smallest exposure whose retained fills
# capture fraction `f` of the wallet's train copyable PnL; f == 1 keeps
# everything). The same train-fit thresholds are applied to the test fills;
# wallets with no positive train PnL get no threshold and are kept whole.

from polymarket_analysis.visualization.wallet_plots import copyable_exposure_series
from polymarket_analysis.wallet_selection.exposure import wallet_capture_thresholds

df_full["deployed_capital"] = df_full["position"] * df_full["price"]

_copyable_wallets = wallet_cohorts["copyable_group"]["wallet"]
test_fills = df_full[
    (~df_full["is_train"]) & (df_full["wallet"].isin(_copyable_wallets))
].copy()
train_fills = df_full[
    (df_full["is_train"]) & (df_full["wallet"].isin(_copyable_wallets))
].copy()

PNL_TO_QTY = {
    COPYABLE_PNL_BASE: "copyable_qty_5m_100",
    **{f"copyable_pnl_{k}": cfg["qty_col"] for k, cfg in COPYABLE_VARIANTS.items()},
}

for key, qty_col in PNL_TO_QTY.items():
    test_fills[f"{key}_exposure"] = test_fills["price"] * test_fills[qty_col]


_capture = wallet_capture_thresholds(
    train_fills, EXPOSURE_FRACS, EXPOSURE_PNL_COL, "deployed_capital"
)


def _annualized_daily_sharpe(daily: pd.Series, periods_per_year: float = 365.0) -> float:
    if len(daily.dropna()) < 2:
        return float("nan")
    full = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
    s = daily.reindex(full, fill_value=0.0)
    sd = s.std()
    if sd == 0:
        return float("nan")
    return float(s.mean() / sd * np.sqrt(periods_per_year))


EXPOSURE_QUANTILES = [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
CARVE_FRACS = sorted({f for f in EXPOSURE_FRACS if f < 1}, reverse=True)

variant_specs = [("wallet", "trade_pnl", "notional")]
variant_specs += [(c, c, f"{c}_exposure") for c in COPYABLE_PNL_COLS]


def _stats_row(sub: pd.DataFrame, label: str, pnl_col: str, exp_col: str, n_total: int) -> dict:
    total_pnl = sub[pnl_col].sum()
    total_capital = sub[exp_col].sum()
    daily = sub.assign(_d=sub["dt"].dt.floor("D")).groupby("_d")[pnl_col].sum()
    profile = copyable_exposure_series(sub, exp_col, bucket_freq="D")
    exp_qs = {
        f"exp_p{int(q * 100)}": (
            profile["exposure"].quantile(q) if not profile.empty else float("nan")
        )
        for q in EXPOSURE_QUANTILES
    }
    return {
        "variant": label,
        "total_pnl": total_pnl,
        "total_roi": (
            total_pnl / total_capital
            if pd.notna(total_capital) and total_capital > 0
            else float("nan")
        ),
        "sharpe": _annualized_daily_sharpe(daily),
        "kept_rows": len(sub),
        "kept_rows_pct": len(sub) / n_total if n_total else float("nan"),
        **exp_qs,
        "exp_max": profile["exposure"].max() if not profile.empty else float("nan"),
    }


n_total = len(test_fills)
stat_rows = []
for label, pnl_col, exp_col in variant_specs:
    stat_rows.append(_stats_row(test_fills, label, pnl_col, exp_col, n_total))
    for frac in CARVE_FRACS:
        T = test_fills["wallet"].map(
            _capture[_capture["frac"] == frac].set_index("wallet")["threshold"]
        ).fillna(0.0)
        carved = test_fills[
            test_fills["deployed_capital"].notna() & (test_fills["deployed_capital"] >= T)
        ]
        if carved.empty:
            continue
        stat_rows.append(_stats_row(carved, f"{label} (frac {frac:g})", pnl_col, exp_col, n_total))

copyable_group_test_stats = pd.DataFrame(stat_rows)
copyable_group_test_stats


,variant,total_pnl,total_roi,sharpe,kept_rows,kept_rows_pct,exp_p25,exp_p50,exp_p75,exp_p90,exp_p95,exp_p99,exp_max
0,wallet,-29694.33,-0.02,-1.13,1371,1.00,3848.42,10326.56,22361.01,29750.41,42005.61,59553.37,67627.93
1,wallet (frac 0.5),-23689.95,-0.02,-0.92,913,0.67,3172.75,9420.88,20272.61,27713.88,37722.12,57669.90,59080.50
2,wallet (frac 0.2),-19421.33,-0.02,-0.79,705,0.51,3247.50,9311.94,18649.12,27319.77,36025.98,57103.37,57598.15
3,copyable_pnl,-81377.91,-0.18,-3.66,1371,1.00,1186.91,3255.19,7308.37,14250.30,16978.46,26647.08,36745.65
4,copyable_pnl (frac 0.5),-73207.51,-0.20,-3.35,913,0.67,708.84,2620.74,7073.12,12789.76,15899.42,27071.41,35730.46
5,copyable_pnl (frac 0.2),-62813.11,-0.19,-2.95,705,0.51,357.86,2400.00,6531.03,12132.69,14191.70,27116.53,35375.78


In [86]:
MARKET = '0xaabf22a03217285585592f3082f27e2b0f4c1cee2bb4003e84817100bbcf1938'

market_def = mdf.loc[mdf['condition_id'] == MARKET].head(1).squeeze()
print(f"Market: {market_def}")

dfc = df_full[(df_full['condition_id'] == MARKET)
              & (df_full['wallet'].isin(wallets))].copy()
# dfc['bucket'] = dfc['dt'].dt.floor('1h')


dfc = dfc.groupby(['dt', 'wallet', 'side', 'outcome']).agg( 
        pnl=('pnl', 'sum'),
        notional=('notional', 'sum'),
        quantity=('quantity', 'sum'),
        position=('position', 'last'),
        avg_price=('price', lambda x: (x * dfc.loc[x.index, 'quantity']).sum() / dfc.loc[x.index, 'quantity'].sum()),
        copyable_qty_5m_100=('copyable_qty_5m_100', 'sum'),
        copyable_pnl=('copyable_pnl', 'sum'),
)[['pnl', 'position', 'notional', 'quantity', 'avg_price', 'copyable_qty_5m_100', 'copyable_pnl']].reset_index().sort_values(['dt', 'wallet', 'side', 'outcome'])

Market: Empty DataFrame
Columns: [condition_id, end_date_iso, question, tags, primary_tag, winner_token_id, outcome]
Index: []


/var/folders/j8/0dbnwk8n6m933m843h7hb88w0000gn/T/ipykernel_82447/3456545487.py:11: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [87]:
# dominant tags
wallet_fills = df_full[df_full['wallet'].isin(wallet_cohorts['multi_filter']['wallet'])]

dominant_tags = (
    wallet_fills
    .groupby(['wallet', 'primary_tag'], as_index=False)[['quantity', 'pnl']].sum()
    .sort_values(['wallet', 'quantity'], ascending=[True, False])
    .assign(total_qty=lambda df: df.groupby('wallet')['quantity'].transform('sum'))
    .drop_duplicates('wallet')
    .assign(primary_tag_ratio=lambda df: df['quantity'] / df['total_qty'])
    .rename(columns={
        'quantity': 'primary_tag_qty'
    })[['wallet', 'primary_tag', 'primary_tag_qty', 'primary_tag_ratio', 'pnl']]
)

In [88]:
print(len(dominant_tags))
dominant_tags.sort_values('pnl', ascending=False).head(20)

25


,wallet,primary_tag,primary_tag_qty,primary_tag_ratio,pnl
22,0xc34b4901be324718c176100e350211d97b199296,Sports,580202.49,1.00,22920.13
6,0x3a01dc75a904043f93f89bce183b45846ed5bc5e,Sports,158077.10,1.00,17923.70
18,0xad2f5bf835fee93d6e7403a274cdc40f7a2dac6a,Sports,289672.73,1.00,14497.49
17,0x9c4fe54fac3952bbced1da8b3c5301fe6fc20b9e,Sports,60093.68,1.00,13574.04
5,0x2f2b6e3176e4914ccc79111dd9c704904d390f9a,Sports,98918.16,1.00,12782.39
24,0xffe290a74a57db36b06cc88692b16f1cda322957,Sports,40320.24,1.00,10807.17
12,0x7accd72259f3582ea4805386d09b311faed5c5b9,Sports,66868.21,1.00,10291.80
11,0x70a6798edbf21eb25cbb749569796d1e641aa2eb,Sports,144539.21,1.00,8129.16
3,0x24ae647bd3d6fec6ec54e3e02606a0a260804c80,Sports,68109.85,1.00,6928.35
1,0x13e4bf768f4c2005af84d9f23c670ac0bc788b89,Sports,50858.73,1.00,6157.90


In [89]:
watched_wallets = wallet_cohorts['multi_filter'].sort_values("copyable_pnl", key=abs, ascending=False)['wallet'].head(10).to_list()
for w in watched_wallets:
    wallet_df = wallet_vol_train[wallet_vol_train['wallet'] == w]
    wallet_cohorts[w] = wallet_df.copy().reset_index(drop=True)

In [90]:
import plotly.graph_objects as go

from polymarket_analysis.visualization.wallet_plots import copyable_exposure_series
from polymarket_analysis.wallet_selection.exposure import wallet_capture_thresholds

print(f"copyable_group wallets: {len(wallet_cohorts['copyable_group']):,}")
print(f"predicting_group wallets: {len(wallet_cohorts['predicting_group']):,}")

pd.set_option('display.max_rows', 100)

PNL_TO_QTY = {
    COPYABLE_PNL_BASE: "copyable_qty_5m_100",
    **{f"copyable_pnl_{k}": cfg["qty_col"] for k, cfg in COPYABLE_VARIANTS.items()},
}

plot_rows = []
exposure_rows = []
carved_rows = []

# Carve thresholds for the capped-carved lines: fit once on the train fills of
# both groups' wallets (same rule as the cohort plots — fit on the uncapped
# base copyable_pnl; fracs < 1 from EXPOSURE_FRACS).
_group_wallets = {
    w for cn in ('copyable_group', 'predicting_group')
    for w in wallet_cohorts[cn]['wallet']
}
_carve_fit = df_full[
    df_full['is_train']
    & df_full['wallet'].isin(_group_wallets)
    & (df_full['side'] == 'BUY')
    & df_full['deployed_capital'].notna()
][['wallet', EXPOSURE_PNL_COL, 'deployed_capital']]
_group_capture = wallet_capture_thresholds(
    _carve_fit, EXPOSURE_FRACS, EXPOSURE_PNL_COL, 'deployed_capital'
)
for group_label, cohort_name in [('copyable', 'copyable_group'), ('predicting', 'predicting_group')]:
    wallets = set(wallet_cohorts[cohort_name]['wallet'])
    group_trades = df_full[
        (df_full['wallet'].isin(wallets))
        & (~df_full['is_train'])
    ][['dt', 'side', 'price', 'pnl', 'last_condition_trade_ts', 'wallet',
       'notional', 'deployed_capital', 'deployed_capital_frac',
       *COPYABLE_PNL_COLS, *PNL_TO_QTY.values(), *CAPPED_PNL_COLS]].copy()

    # per-fill copy exposure (price × copyable qty, BUY only)
    for pnl_col, qty_col in PNL_TO_QTY.items():
        group_trades[f'{pnl_col}_exposure'] = np.where(
            group_trades[qty_col] > 0,
            group_trades['price'] * group_trades[qty_col],
            np.nan,
        )

    # capped-variant exposures: capital actually deployed under the per-wallet cap
    group_trades['pnl_limited_exposure'] = group_trades['notional'] * group_trades['deployed_capital_frac']
    group_trades['copyable_pnl_limited_exposure'] = np.where(
        group_trades['copyable_qty_5m_100'] > 0,
        group_trades['price'] * group_trades['copyable_qty_5m_100'] * group_trades['deployed_capital_frac'],
        np.nan,
    )

    daily = (
        group_trades
        .assign(plot_dt=group_trades['dt'].dt.floor('D'))
        .groupby('plot_dt', as_index=False)[['pnl', *COPYABLE_PNL_COLS, *CAPPED_PNL_COLS]]
        .sum()
        .sort_values('plot_dt')
        .reset_index(drop=True)
    )
    daily['cum_trade_pnl'] = daily['pnl'].cumsum()
    for c in [*COPYABLE_PNL_COLS, *CAPPED_PNL_COLS]:
        daily[f'cum_{c}'] = daily[c].cumsum()
    daily['wallet_group'] = group_label
    plot_rows.append(daily)

    for c in [*COPYABLE_PNL_COLS, *CAPPED_PNL_COLS]:
        exp = copyable_exposure_series(group_trades, f'{c}_exposure', bucket_freq='D')
        exp['wallet_group'] = group_label
        exp['variant'] = c
        exposure_rows.append(exp)

    # capped-carved: carve the test fills by T_{w,f} (train-fit) and
    # accumulate the capped PnLs on the kept fills
    for frac in sorted({f for f in EXPOSURE_FRACS if f < 1}, reverse=True):
        thr = group_trades['wallet'].map(
            _group_capture[_group_capture['frac'] == frac].set_index('wallet')['threshold']
        ).fillna(0.0)
        carved = group_trades[
            group_trades['deployed_capital'].notna()
            & (group_trades['deployed_capital'] >= thr)
        ]
        if carved.empty:
            continue
        cdaily = (
            carved
            .assign(plot_dt=carved['dt'].dt.floor('D'))
            .groupby('plot_dt', as_index=False)[CAPPED_PNL_COLS]
            .sum()
            .sort_values('plot_dt')
            .reset_index(drop=True)
        )
        for c in CAPPED_PNL_COLS:
            cdaily[f'cum_{c}'] = cdaily[c].cumsum()
        cdaily['wallet_group'] = group_label
        cdaily['frac'] = frac
        carved_rows.append(cdaily)

group_perf = pd.concat(plot_rows, ignore_index=True)
group_exposure = pd.concat(exposure_rows, ignore_index=True)
carved_capped = pd.concat(carved_rows, ignore_index=True)

fig = go.Figure()
colors = {'copyable': '#1f77b4', 'predicting': '#ff7f0e'}
dashes = ['dash', 'dot', 'dashdot']
for group_label in ['copyable', 'predicting']:
    g = group_perf[group_perf['wallet_group'] == group_label]
    fig.add_trace(
        go.Scatter(
            x=g['plot_dt'],
            y=g['cum_trade_pnl'],
            mode='lines',
            name=f'{group_label} group trade pnl',
            line=dict(color=colors[group_label], width=2),
        )
    )
    for i, c in enumerate(COPYABLE_PNL_COLS):
        fig.add_trace(
            go.Scatter(
                x=g['plot_dt'],
                y=g[f'cum_{c}'],
                mode='lines',
                name=f'{group_label} {c} (raw, unlimited)',
                line=dict(color=colors[group_label], width=2, dash=dashes[i % len(dashes)]),
            )
        )
    for c in CAPPED_PNL_COLS:
        fig.add_trace(
            go.Scatter(
                x=g['plot_dt'],
                y=g[f'cum_{c}'],
                mode='lines',
                name=f'{group_label} {c} (capped)',
                line=dict(color=colors[group_label], width=2, dash='longdash'),
            )
        )
    for frac in sorted({f for f in EXPOSURE_FRACS if f < 1}, reverse=True):
        cg = carved_capped[
            (carved_capped['wallet_group'] == group_label)
            & (carved_capped['frac'] == frac)
        ]
        for c in CAPPED_PNL_COLS:
            fig.add_trace(
                go.Scatter(
                    x=cg['plot_dt'],
                    y=cg[f'cum_{c}'],
                    mode='lines',
                    name=f'{group_label} {c} (capped, frac {frac:g})',
                    line=dict(color=colors[group_label], width=1.5, dash='dashdot'),
                    opacity=0.8,
                )
            )
    for c in [*COPYABLE_PNL_COLS, *CAPPED_PNL_COLS]:
        e = group_exposure[
            (group_exposure['wallet_group'] == group_label)
            & (group_exposure['variant'] == c)
        ]
        fig.add_trace(
            go.Scatter(
                x=e['dt'],
                y=e['exposure'],
                mode='lines',
                name=f'{group_label} {c} exposure',
                line=dict(color=colors[group_label], width=1.5, dash='dot'),
                opacity=0.75,
                yaxis='y2',
            )
        )

fig.update_layout(
    title='Stage1 wallet-group performance on test period — copyable PnL raw/unlimited + capped (incl. carved fracs), exposure on right axis',
    xaxis_title='Date',
    yaxis_title='Cumulative PnL (USDC)',
    yaxis2=dict(title='Exposure (USDC)', overlaying='y', side='right', showgrid=False),
    template='plotly_white',
)
fig.show()

copyable_group wallets: 25
predicting_group wallets: 14


In [91]:
copyable_wallets = wallet_cohorts['copyable_group'].copy().assign(wallet_group='copyable')
predicting_wallets = wallet_cohorts['predicting_group'].copy().assign(wallet_group='predicting')

selected_wallets = (
    pd.concat([copyable_wallets, predicting_wallets], ignore_index=True)
    .drop_duplicates(subset=['wallet', 'wallet_group'])
    .reset_index(drop=True)
)

selected_wallets.to_parquet(WORKSPACE_DIR / 'selected_wallets_v2.parquet', index=False)
print(f"Saved selected wallets: {len(selected_wallets):,}")
print(selected_wallets['wallet_group'].value_counts().to_dict())


Saved selected wallets: 39
{'copyable': 25, 'predicting': 14}


In [92]:
len(df_wallets[df_wallets['is_train'] == False])

1371

## Wallet PnL over time

Three independent plots:
- **Training plot** — cohort aggregate cumulative PnL over the training period only
- **Test plot** — cohort aggregate cumulative PnL over the test period only (starts from 0)
- **Individual plot** — per-wallet cumulative PnL spanning train + test, with a train/test split
  vline and wallet address labels; test portion resets to 0 at the split boundary

Set `PLOT_WALLET_PNL = False` to skip (e.g. when running headlessly).


In [93]:
PLOT_WALLET_PNL  = True
TOP_N_INDIVIDUAL = 10   # wallets shown in panel 1 per cohort


In [94]:
df_fills = df_full.copy()
df_fills["end_date_iso"] = pd.to_datetime(df_fills["last_condition_trade_ts"], utc=True).dt.floor("D")

# per-variant fill exposure: capital needed to fill the copyable qty at the fill price
PNL_TO_QTY = {
    COPYABLE_PNL_BASE: "copyable_qty_5m_100",
    **{f"copyable_pnl_{k}": cfg["qty_col"] for k, cfg in COPYABLE_VARIANTS.items()},
}
for pnl_col, qty_col in PNL_TO_QTY.items():
    df_fills[f"{pnl_col}_exposure"] = np.where(
        df_fills[qty_col] > 0,
        df_fills["price"] * df_fills[qty_col],
        np.nan,
    )

# capped-variant exposures: capital actually deployed under the per-wallet
# cap (pnl_limited_exposure == min(notional, deployed_capital_limit))
df_fills["pnl_limited_exposure"] = df_fills["notional"] * df_fills["deployed_capital_frac"]
df_fills["copyable_pnl_limited_exposure"] = np.where(
    df_fills["copyable_qty_5m_100"] > 0,
    df_fills["price"] * df_fills["copyable_qty_5m_100"] * df_fills["deployed_capital_frac"],
    np.nan,
)

# Normalise grouped schema → canonical fill-level names
if "total_quantity" in df_fills.columns and "quantity" not in df_fills.columns:
    df_fills = df_fills.rename(columns={
        "total_quantity": "quantity",
        "avg_price": "price",
        "trade_value_usdc": "usdc_amount",
    })

df_fills["usdc_amount"]      = df_fills["usdc_amount"].astype(float)
df_fills["final_value_usdc"] = df_fills["final_value_usdc"].astype(float)
df_fills["quantity"]         = df_fills["quantity"].astype(float)


In [95]:
filter_fills = df_fills[
    (df_fills['wallet'].isin(wallet_cohorts['multi_filter']['wallet']))
    & (df_fills['side'] == 'BUY')
    # & (TAGS is None or df_fills['primary_tag'].isin(TAGS))
    & (df_fills["is_train"] == False)
    ].copy().reset_index(drop=True)

print(len(filter_fills))
# filter_fills = filter_fills[filter_fills['copyable_qty_5m_100'] >= 1].copy().reset_index(drop=True)
print(len(filter_fills))
filter_fills.head(2)

1371
1371


,wallet,condition_id,token_id,dt,side,position,quantity,price,usdc_amount,final_value_usdc,trade_pnl,copyable_pnl,token_winner,final_price,last_condition_trade_ts,tx_hash,num_fills,is_train,copyable_qty_5m_100,avail_copy_qty_5m_100,avail_copy_total_vol_5m_100,avail_copy_count_5m_100,end_date_iso,question,tags,primary_tag,winner_token_id,outcome,pnl,notional,deployed_capital_limit,deployed_capital_frac,pnl_limited,copyable_pnl_limited,deployed_capital,is_carved,copyable_qty_2x,copyable_pnl_2x,copyable_qty_0.5x,copyable_pnl_0.5x,copyable_notional,roi,copyable_roi,bucket,is_copyable,copyable_pnl_exposure,pnl_limited_exposure,copyable_pnl_limited_exposure
0,0xc34b4901be324718c176100e350211d97b199296,0x041eb404c839cdb46387c06c4c74cbc72eb35a0d68adf2ce6171523b64d16788,61700590960124092288395758006899454732638330351977933351529941206161043040947,2026-08-04 22:17:08+00:00,BUY,4457.53,2051.28,0.39,800.00,2051.28,1251.28,126.72,True,1.00,2026-08-05 03:26:26+00:00,0x3fb5ae0a72ca10bb163ce82d55ada2728f3c4a21bafe3f536fd48a046044b8de,1,False,207.74,207.74,126.72,5.00,2026-08-05 00:00:00+00:00,National Bank Open: Luciano Darderi vs Gabriel Diallo,"[Sports, Tennis, Games]",Sports,61700590960124092288395758006899454732638330351977933351529941206161043040947,Luciano Darderi,1251.28,800.00,7353.99,1.00,1251.28,126.72,1738.44,True,21.04,12.83,21.04,12.83,81.02,1.56,1.56,2026-08-04 22:15:00+00:00,True,81.02,800.00,81.02
1,0x5c2311acccc5e9a172f0ce23d4480177bc00cab3,0x0b004edca54b3c30dc7739f24dd9354c93f36bee9945494e83cd7c6dabd5eb75,102306449117315576801306832649246031604275801069051924847210907842838598307729,2026-08-14 06:32:03+00:00,BUY,7589.99,1930.00,0.69,1331.70,0.00,-1331.70,-0.00,False,0.00,2026-08-14 11:24:21+00:00,0x2157386cf180e44d0bc2a51bdbd4c19ebf8c32b965b3428492eff56855f7d9b6,1,False,-0.00,-0.00,0.00,0.00,2026-08-14 00:00:00+00:00,Valorant: Nova Esports vs JD Gaming (BO3) - VCT China Stage 2 Playoffs,"[Sports, Esports, Games, Valorant]",Sports,109900169325103376203882332779793008469455426124016186447448823930349234835209,Nova Esports,-1331.70,1331.70,2559.51,1.00,-1331.70,-0.00,5237.10,True,0.00,0.00,0.00,0.00,0.00,-1.00,NaN,2026-08-14 06:30:00+00:00,False,NaN,1331.70,NaN


In [96]:
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', 1000)

In [97]:
filter_fills["bucket"] = filter_fills["dt"].dt.floor("5min")

MAX_EXPOSURE = 100000

df = filter_fills.copy()

# scale each copyable variant to the per-fill MAX_EXPOSURE capital budget
for c in COPYABLE_PNL_COLS:
    exp_col = f"{c}_exposure"
    scale = np.minimum(1, MAX_EXPOSURE / df[exp_col].replace(0, np.nan))
    df[f"scaled_{c}"] = df[c] * scale
    df[f"scaled_{exp_col}"] = df[exp_col] * scale

df.head(1)


,wallet,condition_id,token_id,dt,side,position,quantity,price,usdc_amount,final_value_usdc,trade_pnl,copyable_pnl,token_winner,final_price,last_condition_trade_ts,tx_hash,num_fills,is_train,copyable_qty_5m_100,avail_copy_qty_5m_100,avail_copy_total_vol_5m_100,avail_copy_count_5m_100,end_date_iso,question,tags,primary_tag,winner_token_id,outcome,pnl,notional,deployed_capital_limit,deployed_capital_frac,pnl_limited,copyable_pnl_limited,deployed_capital,is_carved,copyable_qty_2x,copyable_pnl_2x,copyable_qty_0.5x,copyable_pnl_0.5x,copyable_notional,roi,copyable_roi,bucket,is_copyable,copyable_pnl_exposure,pnl_limited_exposure,copyable_pnl_limited_exposure,scaled_copyable_pnl,scaled_copyable_pnl_exposure
0,0xc34b4901be324718c176100e350211d97b199296,0x041eb404c839cdb46387c06c4c74cbc72eb35a0d68adf2ce6171523b64d16788,61700590960124092288395758006899454732638330351977933351529941206161043040947,2026-08-04 22:17:08+00:00,BUY,4457.53,2051.28,0.39,800.00,2051.28,1251.28,126.72,True,1.00,2026-08-05 03:26:26+00:00,0x3fb5ae0a72ca10bb163ce82d55ada2728f3c4a21bafe3f536fd48a046044b8de,1,False,207.74,207.74,126.72,5.00,2026-08-05 00:00:00+00:00,National Bank Open: Luciano Darderi vs Gabriel Diallo,"[Sports, Tennis, Games]",Sports,61700590960124092288395758006899454732638330351977933351529941206161043040947,Luciano Darderi,1251.28,800.00,7353.99,1.00,1251.28,126.72,1738.44,True,21.04,12.83,21.04,12.83,81.02,1.56,1.56,2026-08-04 22:15:00+00:00,True,81.02,800.00,81.02,126.72,81.02


In [98]:
# ── exposure + PnL over time, per copyable variant ────────────────────────────
agg_dict = {"wallet_pnl": ("trade_pnl", "sum")}
agg_dict.update({f"scaled_{c}": (f"scaled_{c}", "sum") for c in COPYABLE_PNL_COLS})
agg_dict.update({f"scaled_{c}_exposure": (f"scaled_{c}_exposure", "sum") for c in COPYABLE_PNL_COLS})
df_plot = df.groupby("end_date_iso").agg(**agg_dict).reset_index()

df.sort_values("bucket", inplace=True)
df["wallet_pnl_cumsum"] = df["trade_pnl"].cumsum()
for c in COPYABLE_PNL_COLS:
    df[f"scaled_{c}_cumsum"] = df[f"scaled_{c}"].cumsum()
for c in CAPPED_PNL_COLS:
    df[f"{c}_cumsum"] = df[c].cumsum()


def exposure_timeline(sub, c, exp_col=None):
    """Cumulative exposure over time for a subset frame / copyable variant.

    Fills add exposure at buy time; positions resolve at EOD of end_date_iso
    (+1 day), releasing the exposure (negative delta)."""
    if exp_col is None:
        exp_col = f"scaled_{c}_exposure"
    daily = sub.groupby("end_date_iso")[exp_col].sum().reset_index()
    resolution = daily.rename(columns={exp_col: "exposure_delta"})
    resolution["dt"] = pd.to_datetime(resolution["end_date_iso"]) + pd.Timedelta(days=1)
    resolution["exposure_delta"] = -resolution["exposure_delta"]
    resolution["variant"] = c
    buy = sub[["bucket", exp_col]].rename(columns={"bucket": "dt", exp_col: "exposure_delta"})
    buy["variant"] = c
    ev = pd.concat(
        [resolution[["dt", "variant", "exposure_delta"]], buy[["dt", "variant", "exposure_delta"]]],
        ignore_index=True,
    )
    ev.sort_values("dt", inplace=True)
    ev["exposure"] = ev.groupby("variant")["exposure_delta"].cumsum()
    return ev


variant_exposure = pd.concat(
    [exposure_timeline(df, c) for c in COPYABLE_PNL_COLS], ignore_index=True)
capped_exposure = pd.concat(
    [exposure_timeline(df, c, exp_col=f"{c}_exposure") for c in CAPPED_PNL_COLS],
    ignore_index=True)

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df["bucket"],
    y=df["wallet_pnl_cumsum"],
    mode="lines",
    name="wallet_pnl",
))

for c in COPYABLE_PNL_COLS:
    fig.add_trace(go.Scatter(
        x=df["bucket"],
        y=df[f"scaled_{c}_cumsum"],
        mode="lines",
        name=f"scaled_{c}",
        line=dict(dash="dash"),
    ))
for c in CAPPED_PNL_COLS:
    fig.add_trace(go.Scatter(
        x=df["bucket"],
        y=df[f"{c}_cumsum"],
        mode="lines",
        name=f"{c} (capped)",
        line=dict(dash="longdash", width=2.0),
    ))

for c in COPYABLE_PNL_COLS:
    exp_df = variant_exposure[variant_exposure["variant"] == c]
    fig.add_trace(go.Scatter(
        x=exp_df["dt"],
        y=exp_df["exposure"],
        mode="lines",
        name=f"exposure ({c})",
        line=dict(dash="dot"),
    ))
for c in CAPPED_PNL_COLS:
    exp_df = capped_exposure[capped_exposure["variant"] == c]
    fig.add_trace(go.Scatter(
        x=exp_df["dt"],
        y=exp_df["exposure"],
        mode="lines",
        name=f"capped exposure ({c})",
        line=dict(dash="dot"),
    ))

# ── deployed-capital carved copies ───────────────────────────────────────────
# Bankroll-discipline carve: per wallet keep the fills whose deployed capital
# (position * price, post-trade USDC book value on that condition / outcome)
# is >= T_{w,f}, with T fit on TRAIN via wallet_capture_thresholds (the
# smallest exposure capturing fraction `f` of the wallet's train copyable PnL;
# f == 1 keeps everything). The same train-fit T is applied to the test frame
# `df`; PnL / exposure lines are rebuilt on the carved subset.
# frac == 1 is the uncarved copy_all baseline (already drawn above).
import plotly.colors as _pl_colors

df_fills["deployed_capital"] = df_fills["position"] * df_fills["price"]
df["deployed_capital"] = df["position"] * df["price"]

_fit = df_fills[
    df_fills["wallet"].isin(wallet_cohorts["multi_filter"]["wallet"])
    & (df_fills["side"] == "BUY")
    & ((TAGS is None) or df_fills["primary_tag"].isin(TAGS))
    & df_fills["is_train"]
    & df_fills["deployed_capital"].notna()
][["wallet", EXPOSURE_PNL_COL, "deployed_capital"]].copy()

_capture = wallet_capture_thresholds(
    _fit, EXPOSURE_FRACS, EXPOSURE_PNL_COL, "deployed_capital"
)

_exposed_fracs = sorted({f for f in EXPOSURE_FRACS if f < 1}, reverse=True)
FRAC_COLOR = {f: _pl_colors.qualitative.Plotly[i % 10] for i, f in enumerate(_exposed_fracs)}

carve_sizes = {}   # frac -> kept rows (shared across all copyable variants)
carve_index = {}   # frac -> kept index

for frac in _exposed_fracs:
    T = df["wallet"].map(
        _capture[_capture["frac"] == frac].set_index("wallet")["threshold"]
    ).fillna(0.0)
    carved = df[
        df["deployed_capital"].notna() & (df["deployed_capital"] >= T)
    ].sort_values("bucket")
    if carved.empty:
        continue
    color = FRAC_COLOR[frac]
    for c in COPYABLE_PNL_COLS:
        fig.add_trace(go.Scatter(
            x=carved["bucket"],
            y=carved[f"scaled_{c}"].cumsum(),
            mode="lines",
            name=f"scaled_{c} (frac {frac:g})",
            line=dict(color=color, dash="dash", width=2.0),
            opacity=0.95,
        ))
        expos = exposure_timeline(carved, c)
        fig.add_trace(go.Scatter(
            x=expos["dt"],
            y=expos["exposure"],
            mode="lines",
            name=f"exposure ({c}) frac {frac:g}",
            line=dict(color=color, dash="dot", width=1.5),
            opacity=0.75,
        ))
    for c in CAPPED_PNL_COLS:
        fig.add_trace(go.Scatter(
            x=carved["bucket"],
            y=carved[c].cumsum(),
            mode="lines",
            name=f"{c} (capped, frac {frac:g})",
            line=dict(color=color, dash="longdash", width=2.0),
            opacity=0.95,
        ))
        expos = exposure_timeline(carved, c, exp_col=f"{c}_exposure")
        fig.add_trace(go.Scatter(
            x=expos["dt"],
            y=expos["exposure"],
            mode="lines",
            name=f"capped exposure ({c}) frac {frac:g}",
            line=dict(color=color, dash="dot", width=1.5),
            opacity=0.75,
        ))
    carve_sizes[frac] = len(carved)
    carve_index[frac] = carved.index

fig.show(renderer="browser")

# ── validation: how much does each carve actually keep? ─────────────────────
# With train-fit deployed-capital thresholds the carve is shared across all
# copyable variants (same fill set per frac). Show the retained fill share,
# deployed-capital share, and base copyable PnL share on the test frame.
print(f"total test fills (df): {len(df):,}")
validation_rows = []
for frac in _exposed_fracs:
    idx = carve_index.get(frac)
    if idx is None:
        continue
    carved = df.loc[idx]
    validation_rows.append({
        "frac": frac,
        "kept_rows": len(carved),
        "kept_rows_pct": len(carved) / len(df),
        "kept_deployed_cap_pct": (
            carved["deployed_capital"].sum() / df["deployed_capital"].sum()
            if df["deployed_capital"].sum() else float("nan")
        ),
        "kept_copyable_pnl_pct": (
            carved[COPYABLE_PNL_BASE].sum() / df[COPYABLE_PNL_BASE].sum()
            if df[COPYABLE_PNL_BASE].sum() else float("nan")
        ),
    })
pd.DataFrame(validation_rows).round(3)


total test fills (df): 1,371


,frac,kept_rows,kept_rows_pct,kept_deployed_cap_pct,kept_copyable_pnl_pct
0,0.50,913,0.67,0.87,0.90
1,0.20,705,0.51,0.78,0.77


In [99]:
df_fills[(
    (df_fills['is_train'] == False)
    & (df_fills['wallet'].isin(wallet_set))
    & (df_fills['side'] == 'BUY')
          )].groupby(['primary_tag', 'side']).agg(
    copyable_pnl=('copyable_pnl', 'sum'),
    total_copyable_qty=('copyable_qty_5m_100', 'sum'),
    trade_pnl=('trade_pnl', 'sum'),
    trades=('condition_id', 'count'),
    condition_count=('condition_id', 'nunique'),
    wallets=('wallet', lambda x: x.nunique()),
).sort_values('copyable_pnl', key=abs, ascending=False).reset_index().head(10)

/var/folders/j8/0dbnwk8n6m933m843h7hb88w0000gn/T/ipykernel_82447/3940907672.py:5: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



,primary_tag,side,copyable_pnl,total_copyable_qty,trade_pnl,trades,condition_count,wallets
0,Sports,BUY,-81377.91,756109.92,-29694.33,1371,470,25.00
1,Sports,SELL,0.00,0.00,0.00,0,0,NaN


In [100]:
from polymarket_analysis.visualization.wallet_plots import (
    plot_wallet_selection_pnl,
    plot_wallet_individual_pnl,
)

if PLOT_WALLET_PNL:
    # Plot the copyable sharpe terciles in place of the combined copyable_group
    plot_cohorts = {k: v for k, v in wallet_cohorts.items() if k != 'copyable_group'}

    # Column order per cohort: base copyable first, its capped variant right
    # after, then the other variants, wallet-capped last. The library draws
    # per cohort: total → raw lines → carved fracs (train-fit thresholds
    # on the base copyable_pnl) → exposure lines last.
    COHORT_PNL_COLS = [
        COPYABLE_PNL_BASE, "copyable_pnl_limited",
        *COPYABLE_PNL_VARIANTS, "pnl_limited",
    ]
    # Cohort aggregate PnL — training period
    fig_train = plot_wallet_selection_pnl(
        df_fills,
        plot_cohorts,
        split_date=TRAIN_END,
        period="train",
        title="Wallet selection cohorts — cohort cumulative PnL (training period, unlimited + capped + carved copyable PnL, exposure)",
        pnl_cols=COHORT_PNL_COLS,
        exposure_fracs=EXPOSURE_FRACS,
    )
    fig_train.show(renderer="browser")

    # Cohort aggregate PnL — test period (starts from 0)
    fig_test = plot_wallet_selection_pnl(
        df_fills,
        plot_cohorts,
        split_date=TRAIN_END,
        period="test",
        title="Wallet selection cohorts — cohort cumulative PnL (test period, unlimited + capped + carved copyable PnL, exposure)",
        pnl_cols=COHORT_PNL_COLS,
        exposure_fracs=EXPOSURE_FRACS,
    )
    fig_test.show(renderer="browser")

    # # Individual wallet lines (train + test) with split vline and labels
    # fig_ind = plot_wallet_individual_pnl(
    #     df_fills,
    #     wallet_cohorts,
    #     split_date=split_dt,
    #     top_n_individual=TOP_N_INDIVIDUAL,
    #     title="Individual wallet cumulative PnL (train + test, resets at split)",
    # )
    # fig_ind.show(renderer="browser")
